---
# 4. Model Training For 23px by 23px (No Augmentation)

In this section, we will be training various Convolutionary Neural Network (CNN) Models from scratch. Subsequently, we will compare all the Models against each other to obtain the Best Model which will subsequently be used to compare between Data Augmentation and No Data Augmentation. Finally, it will be used to Compare to the 101px by 101px Model.

---
## 4.1 Sequential Model Training

In this section, we will be training Sequential which is a Convolutionary Neural Network (CNN) with the Data Input of **23px by 23px** and **No Data Augmentation**. We will be conducting various operations which includes but not limited to, Hyperparameter Tuning (KerasTuner) and Model Evaluation. We will also explore the various methods utilised to ensure the Model is tuned to output a high accuracy. The Formulas for the Sequential Model are indicated below:

---

**Convolution Operation:**
$$
y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x_{i+m, j+n} \cdot w_{m,n}^{(k)} + b^{(k)}
$$
Where:

- $y_{i,j}^{(k)}$: Output at position $(i, j)$ of the $k$-th feature map

- $x_{i+m, j+n}$: Input pixel value

- $w_{m,n}^{(k)}$: Weight at position $(m, n)$ in the $k$-th filter

- $b^{(k)}$: Bias for filter $k$

- $M \times N$: Size of the convolution kernel (e.g. $3 \times 3$)

---
**Activation Function (ReLU):**

$$
a_{i,j}^{(k)} = \max(0, y_{i,j}^{(k)})
$$
Where:

- $a_{i,j}^{(k)}$: Activated output (after ReLU)

- ReLU discards negative values to introduce non-linearity

---
**Output Layer (Softmax)**

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}
$$
Where:

- $\hat{y}_i$: Probability of class $i$

- $z_i$: Logit output before softmax

- $C$: Total number of classes

---
**Loss Function (Categorical Cossentropy)**

$$L = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)$$
Where:

- $L$: Total loss

- $y_i$: Ground truth label (1 for correct class, else 0)

- $\hat{y}_i$: Predicted probability for class $i$

---
With the Essential Formulas for CNN listed, we will proceed to start preparations for Training the Model in the Code Cells within this Section.

---
### 4.1.1 Defining Callbacks

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the Model Accuracy and optimise the Computation Cost for training each model. These Callbacks will be main used during the Keras Tuner in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 3 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Define the Callbacks in the Code Cell below in preparation for Model Training.

In [ ]:
# ======================== Defining Learning Rate Scheduler ======================== #
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch:
        return lr * 0.5
    return lr

# ======================== Defining Callbacks ======================== #
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
lr_scheduler = LearningRateScheduler(scheduler)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a high accuracy model training.

---
### 4.1.2 Defining Model Parameters

In this sub-section, we will be defining the Model's Parameters that will be tuned using Keras Tuner. There are various Parameters and Characteristics within the Code Cell below. All of these Parameters and their purpose have been indicated below:

---
**L2 Regularisation (Weight Decay)**

Purpose: Penalise Large Weights To Reduce Overfitting.

$$
\text{L2 Penalty} = \lambda \sum_{i=1}^{n} w_i^2
$$

**Where:**

- $λ$: Regularisation Coefficient (e.g., 0.001)

- $w_i$: Weight Parameter

---
**Convolutional Layer**

Purpose: Extract Spatial Features Using Learnable Filters.

$$
Y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} W_{m,n}^{(k)} \cdot X_{i+m,j+n} + b^{(k)}
$$

**Where:**

- $Y_{i,j}^{(k)}$: Output feature map at position $(i,j)$

- $W^{(k)}$: Weights of the $k$-th Filter

- $X$: Input Patch

- $b^{(k)}$: Bias Term

---
**Batch Normalisation**

Purpose: Normalise Layer Activations to Speed Up & Stabilise Training.

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta
$$

**Where:**

- $x$: Activation Value
- $\mu$, $\sigma^2$: Batch Mean and Variance
- $\epsilon$: Small Constant for Numerical Stability
- $\gamma, \beta$: Trainable Parameters (scale and shift)

---
**MaxPooling2D**

Purpose: Reduces Spatial Dimensions while Retaining Important Features.

$$
Y_{i,j} = \max \{X_{2i,2j}, X_{2i,2j+1}, X_{2i+1,2j}, X_{2i+1,2j+1}\}
$$

**Where:**

- $X$: Input feature map
- $Y$: Pooled output

---
**Dropout**

Purpose: Randomly Disables Neurons to Reduce Overfitting.

$$
x_i' =
\begin{cases}
0 & \text{with probability } p \\
\frac{x_i}{1 - p} & \text{otherwise}
\end{cases}
$$

**Where:**

- $ x_i $: Input Neuron
- $ p $: Dropout Rate
- $ x_i' $: Output after Dropout

---
**Dense Layer (Fully Connected)**

Purpose: Learns Global Patterns after Spatial Feature Extraction.

$$
z = \sum_{i=1}^{n} w_i x_i + b, \quad a = \text{ReLU}(z)
$$

**Where:**

- $ w_i $: Weight
- $ x_i $: Input
- $ b $: Bias
- $ \text{ReLU}(z) = \max(0, z) $

---
**Softmax Output Layer**

Purpose: Converts Logits into Class Probabilities.

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**Where:**

- $ z_i $: Logit of Class $ i $
- $ K $: Number of Classes
- $ \sigma(z_i) $: Probability for Class $ i $

---
**Categorical Crossentropy Loss**

Purpose: Computes Distance Between Predicted and True Class Distributions.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

**Where:**

- $ y_i $: Actual Label (one-hot)
- $ \hat{y}_i $: Predicted Probability
- $ K $: Total Number of Classes

---

**Adam Optimiser**

Purpose: Combines Momentum and Adaptive Learning Rate, providing Fast Convergence with Stable Performance.

- First moment estimate:
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
$$

- Second moment estimate:
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left( \nabla L(\theta_t) \right)^2
$$

- Bias correction:
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

**Where:**

- $m_t$: First Moment Estimate (Mean of Gradients)  
- $v_t$: Second Moment Estimate (Uncentered Variance of Gradients)  
- $\hat{m}_t, \hat{v}_t$: Bias-corrected Estimates  
- $\beta_1, \beta_2$: Decay Rates (e.g., $0.9$, $0.999$)  
- $\nabla L(\theta_t)$: Gradient of the Loss at Step $t$  
- $\eta$: Learning Rate  
- $\epsilon$: Small Constant to Prevent Division by Zero  
- $\theta_t$: Model Weights at Step $t$  

---
**RMSprop Optimiser**

Purpose: Adapts Learning Rate Based on a Moving Average of Squared Gradients.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

- Gradient moving average:
$$
v_t = \rho v_{t-1} + (1 - \rho) \left( \nabla L(\theta_t) \right)^2
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{v_t} + \epsilon}
$$

**Where:**

- $v_t$: Running average of squared gradients  
- $\rho$: Decay factor (e.g., $0.9$)  
- $\nabla L(\theta_t)$: Gradient of the loss  
- $\eta$: Learning rate  
- $\epsilon$: Small constant  
- $\theta_t$: Model weights at step $t$  

---

**SGD with Momentum**

Purpose: Accelerates SGD by Accumulating a Velocity Vector in the Direction of Consistent Gradients.

- Velocity update:
$$
v_t = \mu v_{t-1} - \eta \nabla L(\theta_t)
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t + v_t
$$

**Where:**

- $v_t$: Velocity (accumulated gradient)  
- $\mu$: Momentum coefficient (e.g., $0.9$)  
- $\eta$: Learning rate  
- $\nabla L(\theta_t)$: Gradient of loss  
- $\theta_t$: Model weights at step $t$  

---
With the formulas and purpose of each of the Parameter used for the Model listed, we will proceed to define the function in the cell below in preparation for the Model Hyperparameter Search using Keras Tuner.

In [ ]:
# ======================== Defining Model Parameters ======================== #
def build_model(hp):
    model = Sequential()
    # ------ Grayscale Input ----- #
    model.add(Input(shape=(23, 23, 1)))

    # ------ Adding Convolutional Block ----- #
    for i in range(hp.Int("conv_blocks", 1, 3)):
        filters = hp.Choice(f"filters_{i}", [16, 32, 64])
        model.add(Conv2D(
            filters=filters,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_regularizer=l2(0.001)
        ))
        model.add(BatchNormalization())
        model.add(MaxPooling2D())
        model.add(Dropout(hp.Float(f"conv_dropout_{i}", 0.1, 0.4, step=0.1)))

    model.add(Flatten())

    # ----- Adding Fully Connected Layer ----- #
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    model.add(Dense(
        units=dense_units,
        activation='relu',
        kernel_regularizer=l2(0.001)
    ))
    model.add(Dropout(hp.Float("dense_dropout", 0.2, 0.5, step=0.1)))

    # ----- Adding Output Layer ----- #
    model.add(Dense(11, activation='softmax'))

    # ----- Optimiser Selection ----- #
    optimizer_choice = hp.Choice("optimizer", ['adam', 'rmsprop', 'sgd'])
    learning_rate = hp.Choice("lr", [1e-2, 1e-3, 1e-4])

    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_choice == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate, momentum=0.9)

    # ------ Compiling Model ----- #
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

With reference to the Code Cell above, we are able to confirm that the Function for Creating a New Model for the input of 23px by 23px have been successfully defined and it can be utilised for the Keras Tuner.

---
### 4.1.3 Setting Up Keras Tuner

In this sub-section, we will be setting up the Keras Tuner which will be later utilised to find the best hyperparameter for the CNN Model. Setting up the Keras Tuner appropriately will also ensure that the Random Search do not need to be reruned again as a directory can be set up to link the saved Searches from the Google Drive. This will save up Computational Cost and ensure a more efficient and consistent rerun of Codes should the need arise. To allow for replication, the Seed will be set to 42. However, due to varying Computational Capabilities of different GPUs and/or TPUs, a slight deviation will be normal in the various Evaluation Metrics despite having the same Seed.

In [ ]:
# ======================== Setting Up Keras Tuner ======================== #
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='Tunings',
    project_name='kerastuner_23_noaug_sequential',
    overwrite=False,
    seed=42
)

With reference to the Code Cell above, we are able to verify that the Keras Tuner have been established correctly with the directory defined successfully too. We shall move on to conduct the Random Search using the Keras Tuner.

---
### 4.1.4 Conducting Keras Tuner Random Search

In this sub-section, we will be conducting the Random Search to find the best Hyperparameter for the CNN Model using the Keras Tuner. Unlike traditional AIML, there will only be a Random Search and no Grid Search due to the extremely high Computational Resources Require to run a Grid Search for Deep Learning. It is also important to note that the Hyperparameter Random Search will take a long duration due to the nature of the Deep Learning.

In [ ]:
# ======================== Random Searching For Best Hyperparameters ======================== #
tuner.search(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output above, we are able to see that the best Validation Accuracy obtained in 0.908 and the total time taken to conduct the Keras Tuner to find the Best Hyperparameter is around 5 Hours. As the data from the search have been saved, we will not have to rerun the Keras Tuner for this Model again.

---
### 4.1.5 Obtaining Best Hyperparameter

In this sub-section, we will be obtaining the Best Hyperparameter from the Keras Tuner Random Search so as to prepare for the training of the Sequential Model in the next sub-section.

In [ ]:
# ======================== Retrieving Best Model ======================== #
best_hp = tuner.get_best_hyperparameters(1)[0]
model_23_noaug_best = tuner.hypermodel.build(best_hp)

With reference to the Code Cell above, we are able to confirm that the Best Hyperparameter have been obtained and the Model Architecture have been successfully built.

---
### 4.1.6 Sequential Model Checkpoint

In this sub-section, we will be establishing the Model Checkpoint and saving the Model to our Google Drive. It is extremely important to take note that we will only be saving the Best Model so as to prevent excessive storage usage. The key monitor that will be used is the Validation Accuracy so as to achieve the Ideal Fit when placed together with the Validation Data. We will also be only saving the Best Model so as to prevent excessive space usage.

In [ ]:
# ======================== Establishing Model Checkpoint ======================== #
model_dir = "/content/drive/MyDrive/Colab Notebooks/DELE CA1 A/Models"

os.makedirs("/content/drive/MyDrive/Colab Notebooks/DELE CA1 A/Models", exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(model_dir, "best_model_23_noaug_tuned.keras"),
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False
)

With reference to the Code Cell above, we are able to confirm that the Model have been saved successfully to the Google Drive and the Model is ready for Deployment should the need arise.

---
### 4.1.7 Training Sequential Model

In this sub-section, we will be training the Best Sequential Model using the extracted Hyperparameter. This will subsequently allow us to obtain the Model's History which can allow us to conduct various visualisations such as plotting out the Learning Curves to identify the fitting of the Model.

In [ ]:
# ======================== Training Best Model ======================== #
history_23_noaug_best = model_23_noaug_best.fit(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler, checkpoint],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output of the Code Cell above, we are able to infer that the Sequential Model have been successfully trained and will now be ready for Model Evaluation and Model Deployment on the Test Dataset.

---
### 4.1.8 Saving of Model Weights, Model Architecture and Model Tunings

In this sub-section, we will be Saving the Model's Weights into our Google Drive as per the expectaions of the Assessment. The weight will be saved in 'h5' format and will be submitted. The Tunings Record and the Model Architecture will also be saved so as to prevent the need for re-running all the previous codes which will be extremely Computationally Expensive.

In [ ]:
# ==================== Copy Results Back to Drive ==================== #
model_23_noaug_best.save_weights("Weights/weights_23_noaug_sequential_tuned.weights.h5")

shutil.copytree('Tunings',
                os.path.join(DRIVE_ROOT, 'Tunings'),
                dirs_exist_ok=True)

shutil.copytree('Models',
                os.path.join(DRIVE_ROOT, 'Models'),
                dirs_exist_ok=True)

shutil.copytree('Weights',
                os.path.join(DRIVE_ROOT, 'Weights'),
                dirs_exist_ok=True)

With reference to the Code Cell above, we are able to verify that the Sequential Model's Weight have been saved successfully to the Google Drive.

---
### 4.1.9 Model Evaluations

In this sub-section, we will be evaluating the Sequential Model's Performance on the 23px by 23px Data (Not Augmented). We will not be evaluating solely based of the Model's Accuracy, but also based of it's Learning Curve and Specific Images Accuracy. This is to ensure that the Model is able to perform well at a hollistic level and not just solely based of Accuracy or Class Imbalanced Accuracy. More details will be established for the various Evaluations.

---
#### 4.1.9.1 Plotting of Learning Curve

In this sub-section, we will be plotting the Learning Curve to identify the Learning Capabilities of the Sequential Model. This will allow us to identify if the Model is Underfitting, Overfitting or have attained the Ideal Fit. Should the Sequential Model be Underfitting, it means that the Sequential Model is failing to identify and learn the various patterns of the images. Should the Sequential Model be Overfitting, it means that the Model has attained a High Accuracy solely based of Memorising the Data instead of Learning the Data's Attributes. However, if the Sequential Model has a Ideal Fit it is a very good indicator that the Sequential Model has Learnt from the Training Data. The Potential Observations and subsequent Action Plans are indicated below:

**Under Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Over Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Ideal Fitting Learning Curve**
- Accept the Hyperparameter
- Deploy the Model on Testing Data

With the Potential Observations and Action Plan listed, we will proceed to plot out the Learning Curve in the Code Cell below.

In [ ]:
def plot_history(history, title="Sequential Model: 23x23 No-Aug"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # ----- Accuracy Plot ----- #
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # ----- Loss Plot ------ #
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# ======================== Plotting ======================== #
plot_history(history_23_noaug_best)

With reference to the output of the Learning Curve above, we are able to see that the Sequential Model has achieved a **Ideal Learning Curve**. This means that the Sequential Model has indeed learnt from the Training Dataset instead of Memorising or Not Learning at all. With this observation stated, we will now proceed to View the Hyperparameter of the Sequential Model and subsequently, Deploy the Sequential Model on the Testing Data to get the final Evaluation Metrics.

---
#### 4.1.9.2 Model Hyperparameter

In this sub-section, we will be extracting out the Sequential Model's Hyperparameter so as to display the current and Best Hyperparameter for future references within the Project. It is important to take note that the Hyperparameter may be different for other users of the code due to the varying Computational Environment.

In [ ]:
# ======================== Extracting Hyperparameters Into A Dictionary ======================== #
best_hps_dict = best_hp.values

# ======================== Creating of DataFrame ======================== #
hp_df = pd.DataFrame(list(best_hps_dict.items()), columns=['Hyperparameter', 'Value'])

# ======================== Displaying of DataFrame ======================== #
hp_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to view the various parameters of the Sequential Model and we are able to identify the various layers and neurons that makes up the Hidden Layer of the Sequential Model.

---
#### 4.1.9.3 Model Deployment on Testing Data

In this sub-section, we will be Deploying our Sequential Model on the Testing Data provided. Through the Deployment, we are able to obtain a Test Accuracy Metric and also the Test Loss Metric which will allow us to verify the Sequential Model's performance on a Protected Dataset to simulate real-world deployments. The pre-requisite of the Model Performance is evaluated on the following Test Accuracy Threshold:

**Poor**
- Test Accuracy ≤ 9.10...%

**Acceptable**
- Test Accuracy ≥ 20%

**Good**
- Test Accuracy ≥ 50%

**Very Good**
- Test Accuracy ≥ 80%

**Excellent**
- Test Accuracy ≥ 95%

With the threshold for each Performance defined, we will proceed to Deploy the Sequential Model in the Code Cell below to determine the Test Accuracy.

In [ ]:
# ======================== Evaluation On Test Set ======================== #
test_loss, test_acc = model_23_noaug_best.evaluate(test_generator_23, verbose=2)

# ======================== Creating DataFrame ======================== #
results_df = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test Loss'],
    'Value': [test_acc, test_loss]
})

# ======================== Displaying of DataFrame ======================== #
results_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to verify that our Sequential Model has achieved a **Very Good** Test Accuracy of 88.3% and also a Relatively Low Test Loss of 0.57. This is a strong indicator that the Sequential Model has very good Hyperparameters and will most likely perform well.

---
#### 4.1.9.4 Model's Accuracy Confusion Matrix

In this sub-section, we will be plotting the Confusion Matrix of the Sequential Model so as to not only evaluate the Model's Accuracy, but also to identify potential Class Accuracy Weakness and Class Confusion. This will allow us to verify if any actions are required to address these issues and should there be a retuning that needs to be done.

In [ ]:
# ======================== Predicting On Test Data ======================== #
y_pred = np.argmax(model_23_noaug_best.predict(test_generator_23), axis=1)
y_true = test_generator_23.classes
class_labels = list(test_generator_23.class_indices.keys())

# ======================== Plotting Confusion Matrix ======================== #
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)

# ======================== Plot Configurations ======================== #
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=90)
plt.title("Confusion Matrix - Sequential Model (23x23 No-Aug)")
plt.grid(False)
plt.tight_layout()
plt.show()

With reference to the output above, we are able to see that the Sequential Model performed relatively well as seen from the Dark Blue Coloured Squares running from top right to botton left of the Confusion Matrix. However, we are able to see that the Sequential Model does occassionally mixes up some Classes with others such as 'Cabbage' with 'Cauliflower and Broccoli', 'Tomato' with 'Cauliflower and Broccoli' and 'Bitter_Gourd'. However, these Mix-ups are very minor and do not cause a major interference with the Test Accuracy.

---
#### 4.1.9.5 Model's Classification Report

In this sub-section, we will be running a Classification Report for the Sequential Model performance. This Classification Report will give us insights on the various performance of the Sequential Model for each Class instead of just an Overall Test Accuracy. Through this, we can evaluate the performance of the Sequential Model on specific classes and also identify any Weak Classes. The formulas for the various metrics are indicated below:

---
**Precision**

Interpretation: Out of all Predicted Positives, how many are Actually Correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Where:**
- $TP$: True Positives  
- $FP$: False Positives

---
**Recall**

Interpretation: Out of all Actual Positives, how many did we Correctly Identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Where:*
- $TP$: True Positives
- $FN$: False Negatives

---
**F1-Score**

Interpretation: Balances Precision and Recall (useful when Classes are Imbalanced).
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---
**Support**

Interpretation: Total Number of Actual Samples in the Class.

$$
\text{Support} = TP + FN
$$

---
**Accuracy**

Interpretation: Overall Correct Predictions Divided by Total Samples.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Where:**
- $TP$: True Positives
- $TN$: True Negatives
- $FP$: False Positives
- $FN$: False Negatives

---
With the relevant formulas listed, we will proceed to produce the Classification Report in the Code Cell below.

In [ ]:
# ======================== Generating Classification Report ======================== #
report_dict = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)

# ======================== Creating DataFrame ======================== #
report_df = pd.DataFrame(report_dict).transpose()

# ======================== Rounding Off ======================== #
report_df = report_df.round(4)

# ======================== Displaying DataFrame ======================== #
report_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame and the 'F1-Score' Data Column, we are able to see that the F1-Score is relatively high (>80%) throughout the Classes with the exception of Cauliflower and Broccoli. This shows that the Sequential Model generally performs quite well and consistent throughout all the Classes.

---
## 4.2 Functional API Model Training

In this section, we will be training Functional API Model which is a Convolutionary Neural Network (CNN) with the Data Input of **23px by 23px** and **No Data Augmentation**. We will be conducting various operations which includes but not limited to, Hyperparameter Tuning (KerasTuner) and Model Evaluation. We will also explore the various methods utilised to ensure the Model is tuned to output a high accuracy. The Formulas for the Functional API Model are indicated below:

---

**Convolution Operation:**
$$
y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x_{i+m, j+n} \cdot w_{m,n}^{(k)} + b^{(k)}
$$
**Where:**

- $y_{i,j}^{(k)}$: Output at position $(i, j)$ of the $k$-th feature map

- $x_{i+m, j+n}$: Input pixel value

- $w_{m,n}^{(k)}$: Weight at position $(m, n)$ in the $k$-th filter

- $b^{(k)}$: Bias for filter $k$

- $M \times N$: Size of the convolution kernel (e.g. $3 \times 3$)

---
**Activation Function (ReLU):**

$$
a_{i,j}^{(k)} = \max(0, y_{i,j}^{(k)})
$$
Where:

- $a_{i,j}^{(k)}$: Activated output (after ReLU)

- ReLU discards negative values to introduce non-linearity

---
**Max Pooling**
$$
p^{(l)}_{i,j,k} = \max_{\substack{1 \le u \le H_P \\ 1 \le v \le W_P}} a^{(l)}_{s\cdot i+u-1,\, s\cdot j+v-1,\,k}
$$
**Where:**

- $p^{(l)}_{i,j,k}$: Pooled Output

- $a^{(l)}$: Activation Input

- $H_P, W_P$: Pooling Window Size

- $s$: Stride

---
**Dense Layer (Fully Connected)**

$$
z^{(l)}_i = \sum_{j=1}^{n_{in}} W^{(l)}_{ij} \cdot a^{(l-1)}_j + b^{(l)}_i
$$
**Where:**

- $z^{(l)}_i$: Pre-activation of Node $i$

- $a^{(l-1)}_j$: Activation from Previous Layer

- $W^{(l)}_{ij}$: Weight from Node $j$ to $i$

- $b^{(l)}_i$: Bias Term

- $n_{in}$: Number of Input Units

---
**Softmax Output**

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}
$$
**Where:**

- $\hat{y}_i$: Predicted Probability for Class $i$

- $z_i$: Logit for Class $i$

- $K$: Total number of Classes

---
**Categorical Cross-Entropy Loss**

$$
\mathcal{L} = - \sum_{i=1}^K y_i \cdot \log(\hat{y}_i)
$$
**Where:**

- $\mathcal{L}$: Loss Value

- $y_i$: Ground Truth Label (1 or 0)

- $\hat{y}_i$: Predicted Probability

- $K$: Number of Classes

---
With the Essential Formulas for CNN listed, we will proceed to start preparations for Training the Model in the Code Cells within this Section.

---
### 4.2.1 Defining Callbacks

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the Model Accuracy and optimise the Computation Cost for training each model. These Callbacks will be main used during the Keras Tuner in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 3 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Defind the Callbacks in the Code Cell below in preparation for Model Training.

In [ ]:
# ======================== Defining Learning Rate Scheduler ======================== #
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch:
        return lr * 0.5
    return lr

# ======================== Defining Callback ======================== #
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
lr_scheduler = LearningRateScheduler(scheduler)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a high accuracy model training.

---
### 4.2.2 Defining Model Parameters

In this sub-section, we will be defining the Model's Parameters that will be tuned using Keras Tuner. There are various Parameters and Characteristics within the Code Cell below. All of these Parameters and their purpose have been indicated below:

---
**L2 Regularisation (Weight Decay)**

Purpose: Penalise Large Weights To Reduce Overfitting.

$$
\text{L2 Penalty} = \lambda \sum_{i=1}^{n} w_i^2
$$

**Where:**

- $λ$: Regularisation Coefficient (e.g., 0.001)

- $w_i$: Weight Parameter

---
**Convolutional Layer**

Purpose: Extract Spatial Features Using Learnable Filters.

$$
Y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} W_{m,n}^{(k)} \cdot X_{i+m,j+n} + b^{(k)}
$$

**Where:**

- $Y_{i,j}^{(k)}$: Output feature map at position $(i,j)$

- $W^{(k)}$: Weights of the $k$-th Filter

- $X$: Input Patch

- $b^{(k)}$: Bias Term

---
**Batch Normalisation**

Purpose: Normalise Layer Activations to Speed Up & Stabilise Training.

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta
$$

**Where:**

- $x$: Activation Value
- $\mu$, $\sigma^2$: Batch Mean and Variance
- $\epsilon$: Small Constant for Numerical Stability
- $\gamma, \beta$: Trainable Parameters (scale and shift)

---
**MaxPooling2D**

Purpose: Reduces Spatial Dimensions while Retaining Important Features.

$$
Y_{i,j} = \max \{X_{2i,2j}, X_{2i,2j+1}, X_{2i+1,2j}, X_{2i+1,2j+1}\}
$$

**Where:**

- $X$: Input feature map
- $Y$: Pooled output

---
**Dropout**

Purpose: Randomly Disables Neurons to Reduce Overfitting.

$$
x_i' =
\begin{cases}
0 & \text{with probability } p \\
\frac{x_i}{1 - p} & \text{otherwise}
\end{cases}
$$

**Where:**

- $ x_i $: Input Neuron
- $ p $: Dropout Rate
- $ x_i' $: Output after Dropout

---
**Dense Layer (Fully Connected)**

Purpose: Learns Global Patterns after Spatial Feature Extraction.

$$
z = \sum_{i=1}^{n} w_i x_i + b, \quad a = \text{ReLU}(z)
$$

**Where:**

- $ w_i $: Weight
- $ x_i $: Input
- $ b $: Bias
- $ \text{ReLU}(z) = \max(0, z) $

---
**Softmax Output Layer**

Purpose: Converts Logits into Class Probabilities.

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**Where:**

- $ z_i $: Logit of Class $ i $
- $ K $: Number of Classes
- $ \sigma(z_i) $: Probability for Class $ i $

---
**Categorical Crossentropy Loss**

Purpose: Computes Distance Between Predicted and True Class Distributions.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

**Where:**

- $ y_i $: Actual Label (one-hot)
- $ \hat{y}_i $: Predicted Probability
- $ K $: Total Number of Classes

---

**Adam Optimiser**

Purpose: Combines Momentum and Adaptive Learning Rate, providing Fast Convergence with Stable Performance.

- First moment estimate:
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
$$

- Second moment estimate:
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left( \nabla L(\theta_t) \right)^2
$$

- Bias correction:
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

**Where:**

- $m_t$: First Moment Estimate (Mean of Gradients)  
- $v_t$: Second Moment Estimate (Uncentered Variance of Gradients)  
- $\hat{m}_t, \hat{v}_t$: Bias-corrected Estimates  
- $\beta_1, \beta_2$: Decay Rates (e.g., $0.9$, $0.999$)  
- $\nabla L(\theta_t)$: Gradient of the Loss at Step $t$  
- $\eta$: Learning Rate  
- $\epsilon$: Small Constant to Prevent Division by Zero  
- $\theta_t$: Model Weights at Step $t$  

---
**RMSprop Optimiser**

Purpose: Adapts Learning Rate Based on a Moving Average of Squared Gradients.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

- Gradient moving average:
$$
v_t = \rho v_{t-1} + (1 - \rho) \left( \nabla L(\theta_t) \right)^2
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{v_t} + \epsilon}
$$

**Where:**

- $v_t$: Running average of squared gradients  
- $\rho$: Decay factor (e.g., $0.9$)  
- $\nabla L(\theta_t)$: Gradient of the loss  
- $\eta$: Learning rate  
- $\epsilon$: Small constant  
- $\theta_t$: Model weights at step $t$  

---

**SGD with Momentum**

Purpose: Accelerates SGD by Accumulating a Velocity Vector in the Direction of Consistent Gradients.

- Velocity update:
$$
v_t = \mu v_{t-1} - \eta \nabla L(\theta_t)
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t + v_t
$$

**Where:**

- $v_t$: Velocity (accumulated gradient)  
- $\mu$: Momentum coefficient (e.g., $0.9$)  
- $\eta$: Learning rate  
- $\nabla L(\theta_t)$: Gradient of loss  
- $\theta_t$: Model weights at step $t$  

---
With the formulas and purpose of each of the Parameter used for the Model listed, we will proceed to define the function in the cell below in preparation for the Model Hyperparameter Search using Keras Tuner.

In [ ]:
# ======================== Defining Model Parameters ======================== #
def build_functional_model(hp):
    # ----- Functional API Input ----- #
    inputs = Input(shape=(23, 23, 1))
    x = inputs

    # ----- Adding Convolutional Blocks ----- #
    for i in range(hp.Int("conv_blocks", 1, 3)):
        filters = hp.Choice(f"filters_{i}", [16, 32, 64])
        # ----- convolution ----- #
        x = Conv2D(
            filters=filters,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_regularizer=l2(0.001)
        )(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D()(x)
        x = Dropout(hp.Float(f"conv_dropout_{i}", 0.1, 0.4, step=0.1))(x)

    x = Flatten()(x)

    # ----- Fully Connected Layer ----- #
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    x = Dense(
        units=dense_units,
        activation='relu',
        kernel_regularizer=l2(0.001)
    )(x)
    x = Dropout(hp.Float("dense_dropout", 0.2, 0.5, step=0.1))(x)

    # ----- Output Layer ----- #
    outputs = Dense(11, activation='softmax')(x)

    model = Model(inputs, outputs)

    # ----- Optimiser Selection ----- #
    optimizer_choice = hp.Choice("optimizer", ['adam', 'rmsprop', 'sgd'])
    lr_choice = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr_choice)
    elif optimizer_choice == 'rmsprop':
        optimizer = RMSprop(learning_rate=lr_choice)
    else:
        optimizer = SGD(learning_rate=lr_choice, momentum=0.9)

    # ----- Compiling Model ----- #
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

With reference to the Code Cell above, we are able to confirm that the Function for Creating a New Model for the input of 23px by 23px have been successfully defined and it can be utilised for the Keras Tuner.

---
### 4.2.3 Setting Up Keras Tuner

In this sub-section, we will be setting up the Keras Tuner which will be later utilised to find the best hyperparameter for the CNN Model. Setting up the Keras Tuner appropriately will also ensure that the Random Search do not need to be reruned again as a directory can be set up to link the saved Searches from the Google Drive. This will save up Computational Cost and ensure a more efficient and consistent rerun of Codes should the need arise. To allow for replication, the Seed will be set to 42. However, due to varying Computational Capabilities of different GPUs and/or TPUs, a slight deviation will be normal in the various Evaluation Metrics despite having the same Seed.

In [ ]:
# ======================== Setting Up Keras Tuner ======================== #
tuner_functional = kt.RandomSearch(
    build_functional_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='Tunings',
    project_name='kerastuner_23_noaug_functional',
    overwrite=False,
    seed=42
)

With reference to the Code Cell above, we are able to verify that the Keras Tuner have been established correctly with the directory defined successfully too. We shall move on to conduct the Random Search using the Keras Tuner.

---
### 4.2.4 Conducting Keras Tuner Random Search

In this sub-section, we will be conducting the Random Search to find the best Hyperparameter for the CNN Model using the Keras Tuner. Unlike traditional AIML, there will only be a Random Search and no Grid Search due to the extremely high Computational Resources Require to run a Grid Search for Deep Learning. It is also important to note that the Hyperparameter Random Search will take a long duration due to the nature of the Deep Learning.

In [ ]:
# ======================== Random Searching For Best Hyperparameters ======================== #
tuner_functional.search(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output above, we are able to see that the best Validation Accuracy obtained in 0.908 and the total time taken to conduct the Keras Tuner to find the Best Hyperparameter is around 5 Hours. As the data from the search have been saved, we will not have to rerun the Keras Tuner for this Model again.

---
### 4.2.5 Obtaining Best Hyperparameter

In this sub-section, we will be obtaining the Best Hyperparameter from the Keras Tuner Random Search so as to prepare for the training of the Functional API Model in the next sub-section.

In [ ]:
# ======================== Retrieving Best Model ======================== #
best_hp_func = tuner_functional.get_best_hyperparameters(1)[0]
model_functional_23_noaug_best = tuner_functional.hypermodel.build(best_hp_func)

With reference to the Code Cell above, we are able to confirm that the Best Hyperparameter have been obtained and the Model Architecture have been successfully built.

---
### 4.1.6 Functional API Model Checkpoint

In this sub-section, we will be establishing the Model Checkpoint and saving the Model to our Google Drive. It is extremely important to take note that we will only be saving the Best Model so as to prevent excessive storage usage. The key monitor that will be used is the Validation Accuracy so as to achieve the Ideal Fit when placed together with the Validation Data. We will also be only saving the Best Model so as to prevent excessive space usage.

In [ ]:
# ======================== Establishing Model Checkpoint ======================== #
model_dir = "/content/drive/MyDrive/Colab Notebooks/DELE CA1/Models"
os.makedirs(model_dir, exist_ok=True)
checkpoint_functional = ModelCheckpoint(
    filepath=os.path.join(model_dir, "best_model_23_noaug_functional.keras"),
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False
)

With reference to the Code Cell above, we are able to confirm that the Model have been saved successfully to the Google Drive and the Model is ready for Deployment should the need arise.

---
### 4.2.7 Training Functional API Model

In this sub-section, we will be training the Best Functional API Model using the extracted Hyperparameter. This will subsequently allow us to obtain the Model's History which can allow us to conduct various visualisations such as plotting out the Learning Curves to identify the fitting of the Model.

In [ ]:
# ======================== Training Best Model ======================== #
history_functional_23_noaug_best = model_functional_23_noaug_best.fit(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler, checkpoint_functional],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output of the Code Cell above, we are able to infer that the Functional API Model have been successfully trained and will now be ready for Model Evaluation and Model Deployment on the Test Dataset.

---
### 4.2.8 Saving of Model Weights, Model Architecture and Model Tunings

In this sub-section, we will be Saving the Model's Weights into our Google Drive as per the expectaions of the Assessment. The weight will be saved in 'h5' format and will be submitted. The Tunings Record and the Model Architecture will also be saved so as to prevent the need for re-running all the previous codes which will be extremely Computationally Expensive.

In [ ]:
# ==================== Copy Results Back to Drive ==================== #
model_functional_23_noaug_best.save_weights("Weights/weights_23_noaug_functional_tuned.weights.h5")

shutil.copytree('Tunings',
                os.path.join(DRIVE_ROOT, 'Tunings'),
                dirs_exist_ok=True)

shutil.copytree('Models',
                os.path.join(DRIVE_ROOT, 'Models'),
                dirs_exist_ok=True)

shutil.copytree('Weights',
                os.path.join(DRIVE_ROOT, 'Weights'),
                dirs_exist_ok=True)

With reference to the Code Cell above, we are able to verify that the Functional API Model's Weight have been saved successfully to the Google Drive.

---
### 4.2.9 Model Evaluations

In this sub-section, we will be evaluating the Functional API Model's Performance on the 23px by 23px Data (Not Augmented). We will not be evaluating solely based of the Model's Accuracy, but also based of it's Learning Curve and Specific Images Accuracy. This is to ensure that the Model is able to perform well at a hollistic level and not just solely based of Accuracy or Class Imbalanced Accuracy. More details will be established for the various Evaluations.

---
#### 4.2.9.1 Plotting of Learning Curve

In this sub-section, we will be plotting the Learning Curve to identify the Learning Capabilities of the Functional API Model. This will allow us to identify if the Model is Underfitting, Overfitting or have attained the Ideal Fit. Should the Functional API Model be Underfitting, it means that the Functional API Model is failing to identify and learn the various patterns of the images. Should the Functional API Model be Overfitting, it means that the Model has attained a High Accuracy solely based of Memorising the Data instead of Learning the Data's Attributes. However, if the Functional API Model has a Ideal Fit it is a very good indicator that the Functional API Model has Learnt from the Training Data. The Potential Observations and subsequent Action Plans are indicated below:

**Under Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Over Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Ideal Fitting Learning Curve**
- Accept the Hyperparameter
- Deploy the Model on Testing Data

With the Potential Observations and Action Plan listed, we will proceed to plot out the Learning Curve in the Code Cell below.

In [ ]:
def plot_history(history, title="Functional API Model: 23x23 No-Aug"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # ----- Accuracy Plot ----- #
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # ----- Loss Plot ------ #
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# ======================== Plotting ======================== #
plot_history(history_functional_23_noaug_best, title="Functional API Model: 23x23 No-Aug")

With reference to the output of the Learning Curve above, we are able to see that the Functional API Model has achieved a **Ideal Learning Curve**. This means that the Functional API Model has indeed learnt from the Training Dataset instead of Memorising or Not Learning at all. With this observation stated, we will now proceed to View the Hyperparameter of the Functional API Model and subsequently, Deploy the Functional API Model on the Testing Data to get the final Evaluation Metrics.

---
#### 4.2.9.2 Model Hyperparameter

In this sub-section, we will be extracting out the Functional API Model's Hyperparameter so as to display the current and Best Hyperparameter for future references within the Project. It is important to take note that the Hyperparameter may be different for other users of the code due to the varying Computational Environment.

In [ ]:
# ======================== Extracting Hyperparameters Into A DataFrame ======================== #
best_hps_func_dict = best_hp_func.values

# ======================== Creating of DataFrame ======================== #
hp_func_df = pd.DataFrame(list(best_hps_func_dict.items()), columns=['Hyperparameter', 'Value'])

# ======================== Displaying of DataFrame ======================== #
hp_func_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to view the various parameters of the Functional API Model and we are able to identify the various layers and neurons that makes up the Hidden Layer of the Functional API Model.

---
#### 4.2.9.3 Model Deployment on Testing Data

In this sub-section, we will be Deploying our Functional API Model on the Testing Data provided. Through the Deployment, we are able to obtain a Test Accuracy Metric and also the Test Loss Metric which will allow us to verify the Functional API Model's performance on a Protected Dataset to simulate real-world deployments. The pre-requisite of the Model Performance is evaluated on the following Test Accuracy Threshold:

**Poor**
- Test Accuracy ≤ 9.10...%

**Acceptable**
- Test Accuracy ≥ 20%

**Good**
- Test Accuracy ≥ 50%

**Very Good**
- Test Accuracy ≥ 80%

**Excellent**
- Test Accuracy ≥ 95%

With the threshold for each Performance defined, we will proceed to Deploy the Functional API Model in the Code Cell below to determine the Test Accuracy.

In [ ]:
# ======================== Evaluation On Test Set ======================== #
test_loss_func, test_acc_func = model_functional_23_noaug_best.evaluate(test_generator_23, verbose=2)

# ======================== Creating of DataFrame ======================== #
results_func_df = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test Loss'],
    'Value':  [test_acc_func, test_loss_func]
})

# ======================== Displaying of DataFrame ======================== #
results_func_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to verify that our Functional API Model has achieved a **Very Good** Test Accuracy of 87.9% and also a Relatively Low Test Loss of 0.59. This is a strong indicator that the Functional API Model has very good Hyperparameters and will most likely perform well.

---
#### 4.2.9.4 Model's Accuracy Confusion Matrix

In this sub-section, we will be plotting the Confusion Matrix of the Functional API Model so as to not only evaluate the Model's Accuracy, but also to identify potential Class Accuracy Weakness and Class Confusion. This will allow us to verify if any actions are required to address these issues and should there be a retuning that needs to be done.

In [ ]:
# ======================== Predicting On Test Data & Confusion Matrix ======================== #
y_pred_func = np.argmax(model_functional_23_noaug_best.predict(test_generator_23), axis=1)
y_true      = test_generator_23.classes
class_labels = list(test_generator_23.class_indices.keys())

# ======================== Plotting Confusion Matrix ======================== #
cm_func = confusion_matrix(y_true, y_pred_func)
disp_func = ConfusionMatrixDisplay(confusion_matrix=cm_func, display_labels=class_labels)

# ======================== Plot Configurations ======================== #
fig, ax = plt.subplots(figsize=(10, 8))
disp_func.plot(ax=ax, cmap='Blues', xticks_rotation=90)
plt.title("Confusion Matrix - Functional API Model (23x23 No-Aug)")
plt.grid(False)
plt.tight_layout()
plt.show()

With reference to the output above, we are able to see that the Functional API Model performed relatively well as seen from the Dark Blue Coloured Squares running from top right to botton left of the Confusion Matrix. However, we are able to see that the Functional API Model does occassionally mixes up some Classes with others such as 'Cabbage' with 'Cauliflower and Broccoli', 'Cabbage' with 'Cauliflower and Broccoli' and 'Bitter_Gourd' with 'Cauliflower and Broccoli'. However, these Mix-ups are very minor and do not cause a major interference with the Test Accuracy.

---
#### 4.2.9.5 Model's Classification Report

In this sub-section, we will be running a Classification Report for the Functional API Model performance. This Classification Report will give us insights on the various performance of the Functional API Model for each Class instead of just an Overall Test Accuracy. Through this, we can evaluate the performance of the Functional API Model on specific classes and also identify any Weak Classes. The formulas for the various metrics are indicated below:

---
**Precision**

Interpretation: Out of all Predicted Positives, how many are Actually Correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Where:**
- $TP$: True Positives  
- $FP$: False Positives

---
**Recall**

Interpretation: Out of all Actual Positives, how many did we Correctly Identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Where:*
- $TP$: True Positives
- $FN$: False Negatives

---
**F1-Score**

Interpretation: Balances Precision and Recall (useful when Classes are Imbalanced).
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---
**Support**

Interpretation: Total Number of Actual Samples in the Class.

$$
\text{Support} = TP + FN
$$

---
**Accuracy**

Interpretation: Overall Correct Predictions Divided by Total Samples.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Where:**
- $TP$: True Positives
- $TN$: True Negatives
- $FP$: False Positives
- $FN$: False Negatives

---
With the relevant formulas listed, we will proceed to produce the Classification Report in the Code Cell below.

In [ ]:
# ======================== Classification Report ======================== #
report_func = classification_report(y_true, y_pred_func, target_names=class_labels, output_dict=True)
report_func_df = pd.DataFrame(report_func).transpose().round(4)
report_func_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame and the 'F1-Score' Data Column, we are able to see that the F1-Score is relatively high (>80%) throughout the Classes with the exception of Cauliflower and Broccoli. This shows that the Functional API Model generally performs quite well and consistent throughout all the Classes.

---
## 4.3 ResNet-like Model Training

In this section, we will be training ResNet-like Model which is a Convolutionary Neural Network (CNN) with the Data Input of **23px by 23px** and **No Data Augmentation**. We will be conducting various operations which includes but not limited to, Hyperparameter Tuning (KerasTuner) and Model Evaluation. We will also explore the various methods utilised to ensure the Model is tuned to output a high accuracy. The Formulas for the ResNet-like Model are indicated below:

---

**Residual Block Output:**
$$
\mathbf{y}^{(l)} = \mathcal{F}(\mathbf{x}^{(l)}, \{W_i\}) + \mathbf{x}^{(l)}
$$

**Where:**

- $\mathbf{y}^{(l)}$: Output of the Residual Block at Layer $l$

- $\mathbf{x}^{(l)}$: Input to the Residual Block

- $\mathcal{F}(\cdot)$: Residual Function (e.g., 2 or 3 stacked conv layers)

- ${W_i}$: Set of Weights in the Residual Path

---
**Residual Function Example (Two Conv-BN-ReLU Layers):**

$$
\mathcal{F}(\mathbf{x}) = \text{ReLU}\left( \text{BN}_2\left( W_2 * \text{ReLU}\left( \text{BN}_1\left( W_1 * \mathbf{x} \right) \right) \right) \right)
$$
Where:

- $\mathcal{F}(\mathbf{x})$: Residual Transformation of Input $\mathbf{x}$

- $W_1, W_2$: Weights of the Two Convolutional Layers

- $*$: Convolution Operation

- $\text{BN}_1$, $\text{BN}_2$: Batch Normalisation Layers

- $\text{ReLU}(\cdot)$: Rectified Linear Unit activation

---
**Final Output (after Global Average Pooling and Dense)**

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$
Where:

- $\hat{y}_i$: Predicted Probability for Class $i$

- $z_i$: Logit Output from the Final Dense Layer

- $K$: Number of Output Classes

---
**Categorical Cross-Entropy Loss**

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \cdot \log(\hat{y}_i)
$$
Where:

- $\mathcal{L}$: Loss function

- $y_i$: Ground truth label (one-hot)

- $\hat{y}_i$: Predicted Softmax Probability

- $K$: Number of Classes

---
With the Essential Formulas for CNN listed, we will proceed to start preparations for Training the Model in the Code Cells within this Section.

---
### 4.3.1 Defining Callbacks

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the Model Accuracy and optimise the Computation Cost for training each model. These Callbacks will be main used during the Keras Tuner in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 3 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Defind the Callbacks in the Code Cell below in preparation for Model Training.

In [ ]:
# ======================== Defining Learning Rate Scheduler ======================== #
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch:
        return lr * 0.5
    return lr

# ======================== Defining Callbacks ======================== #
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
lr_scheduler = LearningRateScheduler(scheduler)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a high accuracy model training.

---
### 4.3.2 Defining Model Parameters

In this sub-section, we will be defining the Model's Parameters that will be tuned using Keras Tuner. There are various Parameters and Characteristics within the Code Cell below. All of these Parameters and their purpose have been indicated below:

---
**L2 Regularisation (Weight Decay)**

Purpose: Penalise Large Weights To Reduce Overfitting.

$$
\text{L2 Penalty} = \lambda \sum_{i=1}^{n} w_i^2
$$

**Where:**

- $λ$: Regularisation Coefficient (e.g., 0.001)

- $w_i$: Weight Parameter

---
**Convolutional Layer**

Purpose: Extract Spatial Features Using Learnable Filters.

$$
Y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} W_{m,n}^{(k)} \cdot X_{i+m,j+n} + b^{(k)}
$$

**Where:**

- $Y_{i,j}^{(k)}$: Output feature map at position $(i,j)$

- $W^{(k)}$: Weights of the $k$-th Filter

- $X$: Input Patch

- $b^{(k)}$: Bias Term

---
**Batch Normalisation**

Purpose: Normalise Layer Activations to Speed Up & Stabilise Training.

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta
$$

**Where:**

- $x$: Activation Value
- $\mu$, $\sigma^2$: Batch Mean and Variance
- $\epsilon$: Small Constant for Numerical Stability
- $\gamma, \beta$: Trainable Parameters (scale and shift)

---
**MaxPooling2D**

Purpose: Reduces Spatial Dimensions while Retaining Important Features.

$$
Y_{i,j} = \max \{X_{2i,2j}, X_{2i,2j+1}, X_{2i+1,2j}, X_{2i+1,2j+1}\}
$$

**Where:**

- $X$: Input feature map
- $Y$: Pooled output

---
**Dropout**

Purpose: Randomly Disables Neurons to Reduce Overfitting.

$$
x_i' =
\begin{cases}
0 & \text{with probability } p \\
\frac{x_i}{1 - p} & \text{otherwise}
\end{cases}
$$

**Where:**

- $ x_i $: Input Neuron
- $ p $: Dropout Rate
- $ x_i' $: Output after Dropout

---
**Dense Layer (Fully Connected)**

Purpose: Learns Global Patterns after Spatial Feature Extraction.

$$
z = \sum_{i=1}^{n} w_i x_i + b, \quad a = \text{ReLU}(z)
$$

**Where:**

- $ w_i $: Weight
- $ x_i $: Input
- $ b $: Bias
- $ \text{ReLU}(z) = \max(0, z) $

---
**Softmax Output Layer**

Purpose: Converts Logits into Class Probabilities.

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**Where:**

- $ z_i $: Logit of Class $ i $
- $ K $: Number of Classes
- $ \sigma(z_i) $: Probability for Class $ i $

---
**Categorical Crossentropy Loss**

Purpose: Computes Distance Between Predicted and True Class Distributions.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

**Where:**

- $ y_i $: Actual Label (one-hot)
- $ \hat{y}_i $: Predicted Probability
- $ K $: Total Number of Classes

---

**Adam Optimiser**

Purpose: Combines Momentum and Adaptive Learning Rate, providing Fast Convergence with Stable Performance.

- First moment estimate:
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
$$

- Second moment estimate:
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left( \nabla L(\theta_t) \right)^2
$$

- Bias correction:
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

**Where:**

- $m_t$: First Moment Estimate (Mean of Gradients)  
- $v_t$: Second Moment Estimate (Uncentered Variance of Gradients)  
- $\hat{m}_t, \hat{v}_t$: Bias-corrected Estimates  
- $\beta_1, \beta_2$: Decay Rates (e.g., $0.9$, $0.999$)  
- $\nabla L(\theta_t)$: Gradient of the Loss at Step $t$  
- $\eta$: Learning Rate  
- $\epsilon$: Small Constant to Prevent Division by Zero  
- $\theta_t$: Model Weights at Step $t$  

---
**RMSprop Optimiser**

Purpose: Adapts Learning Rate Based on a Moving Average of Squared Gradients.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

- Gradient moving average:
$$
v_t = \rho v_{t-1} + (1 - \rho) \left( \nabla L(\theta_t) \right)^2
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{v_t} + \epsilon}
$$

**Where:**

- $v_t$: Running average of squared gradients  
- $\rho$: Decay factor (e.g., $0.9$)  
- $\nabla L(\theta_t)$: Gradient of the loss  
- $\eta$: Learning rate  
- $\epsilon$: Small constant  
- $\theta_t$: Model weights at step $t$  

---

**SGD with Momentum**

Purpose: Accelerates SGD by Accumulating a Velocity Vector in the Direction of Consistent Gradients.

- Velocity update:
$$
v_t = \mu v_{t-1} - \eta \nabla L(\theta_t)
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t + v_t
$$

**Where:**

- $v_t$: Velocity (accumulated gradient)  
- $\mu$: Momentum coefficient (e.g., $0.9$)  
- $\eta$: Learning rate  
- $\nabla L(\theta_t)$: Gradient of loss  
- $\theta_t$: Model weights at step $t$  

---
With the formulas and purpose of each of the Parameter used for the Model listed, we will proceed to define the function in the cell below in preparation for the Model Hyperparameter Search using Keras Tuner.

In [ ]:
# ======================== Defining Model Parameters ======================== #
def build_resnet_model(hp):
    # ----- Residual Block Definition ----- #
    def res_block(x, filters, dropout_rate):
        shortcut = x

        # ----- First Conv ----- #
        y = Conv2D(filters, (3,3), padding='same',
                   activation='relu', kernel_regularizer=l2(0.001))(x)
        y = BatchNormalization()(y)

        # ----- Second Conv ----- #
        y = Conv2D(filters, (3,3), padding='same',
                   activation=None, kernel_regularizer=l2(0.001))(y)
        y = BatchNormalization()(y)

        # ----- Shortcut Projection if Needed ----- #
        if shortcut.shape[-1] != filters:
            shortcut = Conv2D(filters, (1,1), padding='same',
                              activation=None, kernel_regularizer=l2(0.001))(shortcut)
            shortcut = BatchNormalization()(shortcut)

        # ----- Add & Activate ----- #
        y = Add()([shortcut, y])
        y = Activation('relu')(y)

        # ----- Pooling & Dropout ----- #
        y = MaxPooling2D()(y)
        y = Dropout(dropout_rate)(y)

        return y

    # ----- Input ----- #
    inputs = Input(shape=(23, 23, 1))
    x = inputs

    # ----- Initial Conv ----- #
    init_filters = hp.Choice("init_filters", [16, 32, 64])
    x = Conv2D(init_filters, (3,3), padding='same',
               activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)

    # ----- Residual Blocks ----- #
    for i in range(hp.Int("res_blocks", 1, 3)):
        filters      = hp.Choice(f"res_filters_{i}", [16, 32, 64])
        dropout_rate = hp.Float(f"res_dropout_{i}", 0.1, 0.4, step=0.1)
        x = res_block(x, filters, dropout_rate)

    # ----- Flatten & FC ----- #
    x = Flatten()(x)
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    x = Dense(dense_units, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(hp.Float("dense_dropout", 0.2, 0.5, step=0.1))(x)

    # ----- Output ----- #
    outputs = Dense(11, activation='softmax')(x)
    model = Model(inputs, outputs)

    # ----- Optimiser ----- #
    opt_choice = hp.Choice("optimizer", ['adam', 'rmsprop', 'sgd'])
    lr_choice  = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    if opt_choice == 'adam':
        optimizer = Adam(lr_choice)
    elif opt_choice == 'rmsprop':
        optimizer = RMSprop(lr_choice)
    else:
        optimizer = SGD(lr_choice, momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

With reference to the Code Cell above, we are able to confirm that the Function for Creating a New Model for the input of 23px by 23px have been successfully defined and it can be utilised for the Keras Tuner.

---
### 4.3.3 Setting Up Keras Tuner

In this sub-section, we will be setting up the Keras Tuner which will be later utilised to find the best hyperparameter for the CNN Model. Setting up the Keras Tuner appropriately will also ensure that the Random Search do not need to be reruned again as a directory can be set up to link the saved Searches from the Google Drive. This will save up Computational Cost and ensure a more efficient and consistent rerun of Codes should the need arise. To allow for replication, the Seed will be set to 42. However, due to varying Computational Capabilities of different GPUs and/or TPUs, a slight deviation will be normal in the various Evaluation Metrics despite having the same Seed.

In [ ]:
# ======================== Setting Up Keras Tuner ======================== #
tuner_resnet = kt.RandomSearch(
    build_resnet_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='Tunings',
    project_name='kerastuner_23_noaug_resnet',
    overwrite=False,
    seed=42
)

With reference to the Code Cell above, we are able to verify that the Keras Tuner have been established correctly with the directory defined successfully too. We shall move on to conduct the Random Search using the Keras Tuner.

---
### 4.3.4 Conducting Keras Tuner Random Search

In this sub-section, we will be conducting the Random Search to find the best Hyperparameter for the CNN Model using the Keras Tuner. Unlike traditional AIML, there will only be a Random Search and no Grid Search due to the extremely high Computational Resources Require to run a Grid Search for Deep Learning. It is also important to note that the Hyperparameter Random Search will take a long duration due to the nature of the Deep Learning.

In [ ]:
# ======================== Random Searching For Best Hyperparameters ======================== #
tuner_resnet.search(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output above, we are able to see that the best Validation Accuracy obtained in 0.908 and the total time taken to conduct the Keras Tuner to find the Best Hyperparameter is around 5 Hours. As the data from the search have been saved, we will not have to rerun the Keras Tuner for this Model again.

---
### 4.3.5 Obtaining Best Hyperparameter

In this sub-section, we will be obtaining the Best Hyperparameter from the Keras Tuner Random Search so as to prepare for the training of the ResNet-like Model in the next sub-section.

In [ ]:
# ======================== Retrieving Best Model ======================== #
best_hp_resnet = tuner_resnet.get_best_hyperparameters(1)[0]
model_resnet_23_noaug_best = tuner_resnet.hypermodel.build(best_hp_func)

With reference to the Code Cell above, we are able to confirm that the Best Hyperparameter have been obtained and the Model Architecture have been successfully built.

---
### 4.3.6 ResNet-like Model Checkpoint

In this sub-section, we will be establishing the Model Checkpoint and saving the Model to our Google Drive. It is extremely important to take note that we will only be saving the Best Model so as to prevent excessive storage usage. The key monitor that will be used is the Validation Accuracy so as to achieve the Ideal Fit when placed together with the Validation Data. We will also be only saving the Best Model so as to prevent excessive space usage.

In [ ]:
# ======================== Establishing Model Checkpoint ======================== #
checkpoint_resnet = ModelCheckpoint(
    filepath=os.path.join(model_dir, "best_model_23_noaug_resnet.keras"),
    monitor='val_accuracy', save_best_only=True, save_weights_only=False
)

With reference to the Code Cell above, we are able to confirm that the Model have been saved successfully to the Google Drive and the Model is ready for Deployment should the need arise.

---
### 4.3.7 Training ResNet-like Model

In this sub-section, we will be training the Best ResNet-like Model using the extracted Hyperparameter. This will subsequently allow us to obtain the Model's History which can allow us to conduct various visualisations such as plotting out the Learning Curves to identify the fitting of the Model.

In [ ]:
# ======================== Training Best Model ======================== #
history_resnet_23_noaug_best = model_resnet_23_noaug_best.fit(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler, checkpoint_resnet],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output of the Code Cell above, we are able to infer that the ResNet-like Model have been successfully trained and will now be ready for Model Evaluation and Model Deployment on the Test Dataset.

---
### 4.3.8 Saving of Model Weights, Model Architecture and Model Tunings

In this sub-section, we will be Saving the Model's Weights into our Google Drive as per the expectaions of the Assessment. The weight will be saved in 'h5' format and will be submitted. The Tunings Record and the Model Architecture will also be saved so as to prevent the need for re-running all the previous codes which will be extremely Computationally Expensive.

In [ ]:
# ==================== Copy Results Back to Drive ==================== #
model_resnet_23_noaug_best.save_weights("Weights/weights_23_noaug_resnet_tuned.weights.h5")

shutil.copytree('Tunings',
                os.path.join(DRIVE_ROOT, 'Tunings'),
                dirs_exist_ok=True)

shutil.copytree('Models',
                os.path.join(DRIVE_ROOT, 'Models'),
                dirs_exist_ok=True)

shutil.copytree('Weights',
                os.path.join(DRIVE_ROOT, 'Weights'),
                dirs_exist_ok=True)

With reference to the Code Cell above, we are able to verify that the ResNet-like Model's Weight have been saved successfully to the Google Drive.

---
### 4.3.9 Model Evaluations

In this sub-section, we will be evaluating the ResNet-like Model's Performance on the 23px by 23px Data (Not Augmented). We will not be evaluating solely based of the Model's Accuracy, but also based of it's Learning Curve and Specific Images Accuracy. This is to ensure that the Model is able to perform well at a hollistic level and not just solely based of Accuracy or Class Imbalanced Accuracy. More details will be established for the various Evaluations.

---
#### 4.3.9.1 Plotting of Learning Curve

In this sub-section, we will be plotting the Learning Curve to identify the Learning Capabilities of the ResNet-like Model. This will allow us to identify if the Model is Underfitting, Overfitting or have attained the Ideal Fit. Should the ResNet-like Model be Underfitting, it means that the ResNet-like Model is failing to identify and learn the various patterns of the images. Should the ResNet-like Model be Overfitting, it means that the Model has attained a High Accuracy solely based of Memorising the Data instead of Learning the Data's Attributes. However, if the ResNet-like Model has a Ideal Fit it is a very good indicator that the ResNet-like Model has Learnt from the Training Data. The Potential Observations and subsequent Action Plans are indicated below:

**Under Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Over Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Ideal Fitting Learning Curve**
- Accept the Hyperparameter
- Deploy the Model on Testing Data

With the Potential Observations and Action Plan listed, we will proceed to plot out the Learning Curve in the Code Cell below.

In [ ]:
def plot_history(history, title="ResNet-like Model: 23x23 No-Aug"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # ----- Accuracy Plot ----- #
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # ----- Loss Plot ------ #
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# ======================== Plotting ======================== #
plot_history(history_resnet_23_noaug_best)

With reference to the output of the Learning Curve above, we are able to see that the ResNet-like Model has achieved a **Ideal Learning Curve**. This means that the ResNet-like Model has indeed learnt from the Training Dataset instead of Memorising or Not Learning at all. With this observation stated, we will now proceed to View the Hyperparameter of the ResNet-like Model and subsequently, Deploy the ResNet-like Model on the Testing Data to get the final Evaluation Metrics.

---
#### 4.3.9.2 Model Hyperparameter

In this sub-section, we will be extracting out the ResNet-like Model's Hyperparameter so as to display the current and Best Hyperparameter for future references within the Project. It is important to take note that the Hyperparameter may be different for other users of the code due to the varying Computational Environment.

In [ ]:
# ======================== Extracting Hyperparameters Into A DataFrame ======================== #
best_hps_resnet_dict = best_hp_resnet.values

# ======================== Creating of DataFrame ======================== #
hp_resnet_df = pd.DataFrame(list(best_hps_resnet_dict.items()), columns=['Hyperparameter','Value'])

# ======================== Displaying of DataFrame ======================== #
hp_resnet_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to view the various parameters of the ResNet-like Model and we are able to identify the various layers and neurons that makes up the Hidden Layer of the ResNet-like Model.

---
#### 4.3.9.3 Model Deployment on Testing Data

In this sub-section, we will be Deploying our ResNet-like Model on the Testing Data provided. Through the Deployment, we are able to obtain a Test Accuracy Metric and also the Test Loss Metric which will allow us to verify the ResNet-like Model's performance on a Protected Dataset to simulate real-world deployments. The pre-requisite of the Model Performance is evaluated on the following Test Accuracy Threshold:

**Poor**
- Test Accuracy ≤ 9.10...%

**Acceptable**
- Test Accuracy ≥ 20%

**Good**
- Test Accuracy ≥ 50%

**Very Good**
- Test Accuracy ≥ 80%

**Excellent**
- Test Accuracy ≥ 95%

With the threshold for each Performance defined, we will proceed to Deploy the ResNet-like Model in the Code Cell below to determine the Test Accuracy.

In [ ]:
# ======================== Evaluation On Test Set ======================== #
test_loss_res, test_acc_res = model_resnet_23_noaug_best.evaluate(test_generator_23, verbose=2)
results_res_df = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test Loss'],
    'Value':  [test_acc_res, test_loss_res]
})
results_res_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to verify that our ResNet-like Model has achieved a **Good** Test Accuracy of 79.0% and also a Relatively Low Test Loss of 0.81. This is a strong indicator that the ResNet-like Model has very good Hyperparameters and will most likely perform well.

---
#### 4.3.9.4 Model's Accuracy Confusion Matrix

In this sub-section, we will be plotting the Confusion Matrix of the ResNet-like Model so as to not only evaluate the Model's Accuracy, but also to identify potential Class Accuracy Weakness and Class Confusion. This will allow us to verify if any actions are required to address these issues and should there be a retuning that needs to be done.

In [ ]:
# ======================== Predicting On Test Data ======================== #
y_pred_res = np.argmax(model_resnet_23_noaug_best.predict(test_generator_23), axis=1)

# ======================== Plotting Confusion Matrix ======================== #
cm_res = confusion_matrix(test_generator_23.classes, y_pred_res)
disp_res = ConfusionMatrixDisplay(cm_res, display_labels=class_labels)

# ======================== Plot Configurations ======================== #
fig, ax = plt.subplots(figsize=(10,8))
disp_res.plot(ax=ax, cmap='Blues', xticks_rotation=90)
plt.title("Confusion Matrix - ResNet-like Model (23x23 No-Aug)")
plt.tight_layout()
plt.show()

With reference to the output above, we are able to see that the ResNet-like Model performed relatively well as seen from the Dark Blue Coloured Squares running from top right to botton left of the Confusion Matrix. However, we are able to see that the ResNet-like Model does occassionally mixes up some Classes with others such as 'Cabbage' with 'Cauliflower and Broccoli', 'Tomato' with 'Cauliflower and Broccoli' and 'Bitter_Gourd'. It is important to take note that in this case, the Resnet Model performed poorer compared to the other models as the 'mix-ups' can cause a major interference with the Test Accuracy.

---
#### 4.3.9.5 Model's Classification Report

In this sub-section, we will be running a Classification Report for the ResNet-like Model performance. This Classification Report will give us insights on the various performance of the ResNet-like Model for each Class instead of just an Overall Test Accuracy. Through this, we can evaluate the performance of the ResNet-like Model on specific classes and also identify any Weak Classes. The formulas for the various metrics are indicated below:

---
**Precision**

Interpretation: Out of all Predicted Positives, how many are Actually Correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Where:**
- $TP$: True Positives  
- $FP$: False Positives

---
**Recall**

Interpretation: Out of all Actual Positives, how many did we Correctly Identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Where:*
- $TP$: True Positives
- $FN$: False Negatives

---
**F1-Score**

Interpretation: Balances Precision and Recall (useful when Classes are Imbalanced).
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---
**Support**

Interpretation: Total Number of Actual Samples in the Class.

$$
\text{Support} = TP + FN
$$

---
**Accuracy**

Interpretation: Overall Correct Predictions Divided by Total Samples.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Where:**
- $TP$: True Positives
- $TN$: True Negatives
- $FP$: False Positives
- $FN$: False Negatives

---
With the relevant formulas listed, we will proceed to produce the Classification Report in the Code Cell below.

In [ ]:
# ======================== Classification Report ======================== #
report_res = classification_report(test_generator_23.classes, y_pred_res,
                                   target_names=class_labels, output_dict=True)

# ======================== Rounding Off ======================== #
report_res_df = pd.DataFrame(report_res).transpose().round(4)

# ======================== Displaying DataFrame ======================== #
report_res_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame and the 'F1-Score' Data Column, we are able to see that the F1-Score is somewhat high (>75%) throughout the Classes with the exception of Cauliflower and Broccoli, and Cabbage. This shows that the ResNet-like Model generally performs quite well and consistent throughout all the Classes.

---
## 4.4 Inception-like Model Training

In this section, we will be training Inception-like which is a Convolutionary Neural Network (CNN) with the Data Input of **23px by 23px** and **No Data Augmentation**. We will be conducting various operations which includes but not limited to, Hyperparameter Tuning (KerasTuner) and Model Evaluation. We will also explore the various methods utilised to ensure the Model is tuned to output a high accuracy. The Formulas for the Inception-like Model are indicated below:

---

**Inception Module Output (Concatenation of Branches):**
$$
\mathbf{y} = [\mathbf{y}_1 \, \| \, \mathbf{y}_2 \, \| \, \mathbf{y}_3 \, \| \, \mathbf{y}_4]
$$
Where:

- $\mathbf{y}$: Output of the Inception Module

- $\mathbf{y}_1$: Output of $1 \times 1$ Convolution Branch

- $\mathbf{y}_2$: Output of $1 \times 1 \rightarrow 3 \times 3$ Convolution Branch

- $\mathbf{y}_3$: Output of $1 \times 1 \rightarrow 5 \times 5$ Convolution B
Branch

- $\mathbf{y}_4$: Output of Max Pooling $\rightarrow 1 \times 1$ Projection Branch

- $|$: Channel-wise (depth) Concatenation

---
**1×1 Convolution for Dimensionality Reduction:**

$$
a_{i,j}^{(k)} = \max(0, y_{i,j}^{(k)})
$$
Where:

- $\mathbf{y}_1$: Output Feature Map after $1 \times 1$ Convolution

- $W_{1\times1}$: Kernel Weights of $1 \times 1$ Conv

- $\mathbf{x}$: Input Feature Map

- $b$: Bias

- $*$: Convolution Operation

- $\sigma(\cdot)$: Activation Function (e.g., ReLU)

---
**Example of 1×1 → 3×3 Convolutional Path**

$$
\mathbf{y}_2 = \sigma\left(W_{3\times3} * \sigma(W_{1\times1} * \mathbf{x} + b_1) + b_2\right)
$$
Where:

- $\mathbf{y}_2$: Output of the Second Inception Branch

- $W_{1\times1}, W_{3\times3}$: Weights for $1 \times 1$ and $3 \times 3$ Convs

- $b_1, b_2$: Bias Terms

- $\sigma(\cdot)$: Activation Function (e.g., ReLU)

---
**Final Output Softmax**

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Where:

- $\hat{y}_i$: Predicted Class Probability

- $z_i$: Logit for Class $i$

- $K$: Number of Output Classes

---
**Categorical Cross-Entropy Loss**

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \cdot \log(\hat{y}_i)
$$

Where:

- $\mathcal{L}$: Loss value

- $y_i$: Ground truth label

- $\hat{y}_i$: Predicted probability

- $K$: Number of classes
---
With the Essential Formulas for CNN listed, we will proceed to start preparations for Training the Model in the Code Cells within this Section.

---
### 4.4.1 Defining Callbacks

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the Model Accuracy and optimise the Computation Cost for training each model. These Callbacks will be main used during the Keras Tuner in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 3 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Defind the Callbacks in the Code Cell below in preparation for Model Training.

In [ ]:
# ======================== Defining Learning Rate Scheduler ======================== #
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch:
        return lr * 0.5
    return lr

# ======================== Defining Callback ======================== #
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
lr_scheduler = LearningRateScheduler(scheduler)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a high accuracy model training.

---
### 4.4.2 Defining Model Parameters

In this sub-section, we will be defining the Model's Parameters that will be tuned using Keras Tuner. There are various Parameters and Characteristics within the Code Cell below. All of these Parameters and their purpose have been indicated below:

---
**L2 Regularisation (Weight Decay)**

Purpose: Penalise Large Weights To Reduce Overfitting.

$$
\text{L2 Penalty} = \lambda \sum_{i=1}^{n} w_i^2
$$

**Where:**

- $λ$: Regularisation Coefficient (e.g., 0.001)

- $w_i$: Weight Parameter

---
**Convolutional Layer**

Purpose: Extract Spatial Features Using Learnable Filters.

$$
Y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} W_{m,n}^{(k)} \cdot X_{i+m,j+n} + b^{(k)}
$$

**Where:**

- $Y_{i,j}^{(k)}$: Output feature map at position $(i,j)$

- $W^{(k)}$: Weights of the $k$-th Filter

- $X$: Input Patch

- $b^{(k)}$: Bias Term

---
**Batch Normalisation**

Purpose: Normalise Layer Activations to Speed Up & Stabilise Training.

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta
$$

**Where:**

- $x$: Activation Value
- $\mu$, $\sigma^2$: Batch Mean and Variance
- $\epsilon$: Small Constant for Numerical Stability
- $\gamma, \beta$: Trainable Parameters (scale and shift)

---
**MaxPooling2D**

Purpose: Reduces Spatial Dimensions while Retaining Important Features.

$$
Y_{i,j} = \max \{X_{2i,2j}, X_{2i,2j+1}, X_{2i+1,2j}, X_{2i+1,2j+1}\}
$$

**Where:**

- $X$: Input feature map
- $Y$: Pooled output

---
**Dropout**

Purpose: Randomly Disables Neurons to Reduce Overfitting.

$$
x_i' =
\begin{cases}
0 & \text{with probability } p \\
\frac{x_i}{1 - p} & \text{otherwise}
\end{cases}
$$

**Where:**

- $ x_i $: Input Neuron
- $ p $: Dropout Rate
- $ x_i' $: Output after Dropout

---
**Dense Layer (Fully Connected)**

Purpose: Learns Global Patterns after Spatial Feature Extraction.

$$
z = \sum_{i=1}^{n} w_i x_i + b, \quad a = \text{ReLU}(z)
$$

**Where:**

- $ w_i $: Weight
- $ x_i $: Input
- $ b $: Bias
- $ \text{ReLU}(z) = \max(0, z) $

---
**Softmax Output Layer**

Purpose: Converts Logits into Class Probabilities.

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**Where:**

- $ z_i $: Logit of Class $ i $
- $ K $: Number of Classes
- $ \sigma(z_i) $: Probability for Class $ i $

---
**Categorical Crossentropy Loss**

Purpose: Computes Distance Between Predicted and True Class Distributions.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

**Where:**

- $ y_i $: Actual Label (one-hot)
- $ \hat{y}_i $: Predicted Probability
- $ K $: Total Number of Classes

---

**Adam Optimiser**

Purpose: Combines Momentum and Adaptive Learning Rate, providing Fast Convergence with Stable Performance.

- First moment estimate:
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
$$

- Second moment estimate:
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left( \nabla L(\theta_t) \right)^2
$$

- Bias correction:
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

**Where:**

- $m_t$: First Moment Estimate (Mean of Gradients)  
- $v_t$: Second Moment Estimate (Uncentered Variance of Gradients)  
- $\hat{m}_t, \hat{v}_t$: Bias-corrected Estimates  
- $\beta_1, \beta_2$: Decay Rates (e.g., $0.9$, $0.999$)  
- $\nabla L(\theta_t)$: Gradient of the Loss at Step $t$  
- $\eta$: Learning Rate  
- $\epsilon$: Small Constant to Prevent Division by Zero  
- $\theta_t$: Model Weights at Step $t$  

---
**RMSprop Optimiser**

Purpose: Adapts Learning Rate Based on a Moving Average of Squared Gradients.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

- Gradient moving average:
$$
v_t = \rho v_{t-1} + (1 - \rho) \left( \nabla L(\theta_t) \right)^2
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{v_t} + \epsilon}
$$

**Where:**

- $v_t$: Running average of squared gradients  
- $\rho$: Decay factor (e.g., $0.9$)  
- $\nabla L(\theta_t)$: Gradient of the loss  
- $\eta$: Learning rate  
- $\epsilon$: Small constant  
- $\theta_t$: Model weights at step $t$  

---

**SGD with Momentum**

Purpose: Accelerates SGD by Accumulating a Velocity Vector in the Direction of Consistent Gradients.

- Velocity update:
$$
v_t = \mu v_{t-1} - \eta \nabla L(\theta_t)
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t + v_t
$$

**Where:**

- $v_t$: Velocity (accumulated gradient)  
- $\mu$: Momentum coefficient (e.g., $0.9$)  
- $\eta$: Learning rate  
- $\nabla L(\theta_t)$: Gradient of loss  
- $\theta_t$: Model weights at step $t$  

---
With the formulas and purpose of each of the Parameter used for the Model listed, we will proceed to define the function in the cell below in preparation for the Model Hyperparameter Search using Keras Tuner.

In [ ]:
# ======================== Defining Model Parameters ======================== #
def build_inception_model(hp):
    # ----- Inception Block Definition ----- #
    def inception_block(x, filters):
        # ----- 1x1 Conv ----- #
        path1 = Conv2D(filters, (1,1), padding='same', activation='relu',
                       kernel_regularizer=l2(0.001))(x)
        # ----- 3x3 Conv ----- #
        path2 = Conv2D(filters, (3,3), padding='same', activation='relu',
                       kernel_regularizer=l2(0.001))(x)
        # ----- 5x5 Conv ----- #
        path3 = Conv2D(filters, (5,5), padding='same', activation='relu',
                       kernel_regularizer=l2(0.001))(x)
        # ----- Pool + 1x1 ----- #
        path4 = MaxPooling2D((3,3), strides=(1,1), padding='same')(x)
        path4 = Conv2D(filters, (1,1), padding='same', activation='relu',
                       kernel_regularizer=l2(0.001))(path4)
        # ----- Concat ----- #
        return Concatenate()([path1, path2, path3, path4])

    # ----- Input ----- #
    inputs = Input(shape=(23, 23, 1))
    x = inputs

    # ----- Inception Blocks ----- #
    for i in range(hp.Int("inc_blocks", 1, 2)):
        filters = hp.Choice(f"inc_filters_{i}", [16, 32, 64])
        x = inception_block(x, filters)
        x = Dropout(hp.Float(f"inc_dropout_{i}", 0.1, 0.4, step=0.1))(x)

    x = Flatten()(x)

    # ----- Dense Layer ----- #
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    x = Dense(dense_units, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(hp.Float("dense_dropout", 0.2, 0.5, step=0.1))(x)

    # ----- Output ----- #
    outputs = Dense(11, activation='softmax')(x)
    model = Model(inputs, outputs)

    # ----- Optimiser ----- #
    opt = hp.Choice("optimizer", ['adam','rmsprop','sgd'])
    lr  = hp.Choice("lr", [1e-2,1e-3,1e-4])
    if opt=='adam':
        optimizer = Adam(lr)
    elif opt=='rmsprop':
        optimizer = RMSprop(lr)
    else:
        optimizer = SGD(lr, momentum=0.9)

    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

With reference to the Code Cell above, we are able to confirm that the Function for Creating a New Model for the input of 23px by 23px have been successfully defined and it can be utilised for the Keras Tuner.

---
### 4.4.3 Setting Up Keras Tuner

In this sub-section, we will be setting up the Keras Tuner which will be later utilised to find the best hyperparameter for the CNN Model. Setting up the Keras Tuner appropriately will also ensure that the Random Search do not need to be reruned again as a directory can be set up to link the saved Searches from the Google Drive. This will save up Computational Cost and ensure a more efficient and consistent rerun of Codes should the need arise. To allow for replication, the Seed will be set to 42. However, due to varying Computational Capabilities of different GPUs and/or TPUs, a slight deviation will be normal in the various Evaluation Metrics despite having the same Seed.

In [ ]:
# ======================== Setting Up Keras Tuner ======================== #
tuner_inception = kt.RandomSearch(
    build_inception_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='Tunings',
    project_name='kerastuner_23_noaug_inception',
    overwrite=False,
    seed=42
)

With reference to the Code Cell above, we are able to verify that the Keras Tuner have been established correctly with the directory defined successfully too. We shall move on to conduct the Random Search using the Keras Tuner.

---
### 4.4.4 Conducting Keras Tuner Random Search

In this sub-section, we will be conducting the Random Search to find the best Hyperparameter for the CNN Model using the Keras Tuner. Unlike traditional AIML, there will only be a Random Search and no Grid Search due to the extremely high Computational Resources Require to run a Grid Search for Deep Learning. It is also important to note that the Hyperparameter Random Search will take a long duration due to the nature of the Deep Learning.

In [ ]:
# ======================== Random Searching For Best Hyperparameters ======================== #
tuner_inception.search(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output above, we are able to see that the best Validation Accuracy obtained in 0.908 and the total time taken to conduct the Keras Tuner to find the Best Hyperparameter is around 5 Hours. As the data from the search have been saved, we will not have to rerun the Keras Tuner for this Model again.

---
### 4.4.5 Obtaining Best Hyperparameter

In this sub-section, we will be obtaining the Best Hyperparameter from the Keras Tuner Random Search so as to prepare for the training of the Inception-like Model in the next sub-section.

In [ ]:
# ======================== Retrieving Best Model ======================== #
best_hp_inc = tuner_inception.get_best_hyperparameters(1)[0]
model_inception_23_noaug_best = tuner_inception.hypermodel.build(best_hp_inc)

With reference to the Code Cell above, we are able to confirm that the Best Hyperparameter have been obtained and the Model Architecture have been successfully built.

---
### 4.4.6 Inception-like Model Checkpoint

In this sub-section, we will be establishing the Model Checkpoint and saving the Model to our Google Drive. It is extremely important to take note that we will only be saving the Best Model so as to prevent excessive storage usage. The key monitor that will be used is the Validation Accuracy so as to achieve the Ideal Fit when placed together with the Validation Data. We will also be only saving the Best Model so as to prevent excessive space usage.

In [ ]:
# ======================== Checkpoint & Training ======================== #
checkpoint_inception = ModelCheckpoint(
    filepath=os.path.join(model_dir, "best_model_23_noaug_inception.keras"),
    monitor='val_accuracy', save_best_only=True
)

With reference to the Code Cell above, we are able to confirm that the Model have been saved successfully to the Google Drive and the Model is ready for Deployment should the need arise.

---
### 4.4.7 Training Inception-like Model

In this sub-section, we will be training the Best Inception-like Model using the extracted Hyperparameter. This will subsequently allow us to obtain the Model's History which can allow us to conduct various visualisations such as plotting out the Learning Curves to identify the fitting of the Model.

In [ ]:
# ======================== Training Best Model ======================== #
history_inception_23_noaug_best = model_inception_23_noaug_best.fit(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler, checkpoint_inception],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output of the Code Cell above, we are able to infer that the Inception-like Model have been successfully trained and will now be ready for Model Evaluation and Model Deployment on the Test Dataset.

---
### 4.4.8 Saving of Model Weights, Model Architecture and Model Tunings

In this sub-section, we will be Saving the Model's Weights into our Google Drive as per the expectaions of the Assessment. The weight will be saved in 'h5' format and will be submitted. The Tunings Record and the Model Architecture will also be saved so as to prevent the need for re-running all the previous codes which will be extremely Computationally Expensive.

In [ ]:
# ==================== Copy Results Back to Drive ==================== #
model_inception_23_noaug_best.save_weights("Weights/weights_23_noaug_inception_tuned.weights.h5")

shutil.copytree('Tunings',
                os.path.join(DRIVE_ROOT, 'Tunings'),
                dirs_exist_ok=True)

shutil.copytree('Models',
                os.path.join(DRIVE_ROOT, 'Models'),
                dirs_exist_ok=True)

shutil.copytree('Weights',
                os.path.join(DRIVE_ROOT, 'Weights'),
                dirs_exist_ok=True)

With reference to the Code Cell above, we are able to verify that the Inception-like Model's Weight have been saved successfully to the Google Drive.

---
### 4.4.9 Model Evaluations

In this sub-section, we will be evaluating the Inception-like Model's Performance on the 23px by 23px Data (Not Augmented). We will not be evaluating solely based of the Model's Accuracy, but also based of it's Learning Curve and Specific Images Accuracy. This is to ensure that the Model is able to perform well at a hollistic level and not just solely based of Accuracy or Class Imbalanced Accuracy. More details will be established for the various Evaluations.

---
#### 4.4.9.1 Plotting of Learning Curve

In this sub-section, we will be plotting the Learning Curve to identify the Learning Capabilities of the Inception-like Model. This will allow us to identify if the Model is Underfitting, Overfitting or have attained the Ideal Fit. Should the Inception-like Model be Underfitting, it means that the Inception-like Model is failing to identify and learn the various patterns of the images. Should the Inception-like Model be Overfitting, it means that the Model has attained a High Accuracy solely based of Memorising the Data instead of Learning the Data's Attributes. However, if the Inception-like Model has a Ideal Fit it is a very good indicator that the Inception-like Model has Learnt from the Training Data. The Potential Observations and subsequent Action Plans are indicated below:

**Under Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Over Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Ideal Fitting Learning Curve**
- Accept the Hyperparameter
- Deploy the Model on Testing Data

With the Potential Observations and Action Plan listed, we will proceed to plot out the Learning Curve in the Code Cell below.

In [ ]:
def plot_history(history, title="Inception-like Model: 23x23 No-Aug"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # ----- Accuracy Plot ----- #
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # ----- Loss Plot ------ #
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# ======================== Plotting ======================== #
plot_history(history_inception_23_noaug_best)

With reference to the output of the Learning Curve above, we are able to see that the Inception-like Model has achieved a **Ideal Learning Curve**. This means that the Inception-like Model has indeed learnt from the Training Dataset instead of Memorising or Not Learning at all. However, we should also we concerned that there is a significant gap between the Training Accuracy and Validation Accuracy. With this observation stated, we will now proceed to View the Hyperparameter of the Inception-like Model and subsequently, Deploy the Inception-like Model on the Testing Data to get the final Evaluation Metrics.

---
#### 4.4.9.2 Model Hyperparameter

In this sub-section, we will be extracting out the Inception-like Model's Hyperparameter so as to display the current and Best Hyperparameter for future references within the Project. It is important to take note that the Hyperparameter may be different for other users of the code due to the varying Computational Environment.

In [ ]:
# ======================== Hyperparameter DataFrame ======================== #
best_hps_inc_dict = best_hp_inc.values

# ======================== Creating DataFrame ======================== #
hp_inc_df = pd.DataFrame(list(best_hps_inc_dict.items()), columns=['Hyperparameter','Value'])

# ======================== Displaying DataFrame ======================== #
hp_inc_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to view the various parameters of the Inception-like Model and we are able to identify the various layers and neurons that makes up the Hidden Layer of the Inception-like Model.

---
#### 4.4.9.3 Model Deployment on Testing Data

In this sub-section, we will be Deploying our Inception-like Model on the Testing Data provided. Through the Deployment, we are able to obtain a Test Accuracy Metric and also the Test Loss Metric which will allow us to verify the Inception-like Model's performance on a Protected Dataset to simulate real-world deployments. The pre-requisite of the Model Performance is evaluated on the following Test Accuracy Threshold:

**Poor**
- Test Accuracy ≤ 9.10...%

**Acceptable**
- Test Accuracy ≥ 20%

**Good**
- Test Accuracy ≥ 50%

**Very Good**
- Test Accuracy ≥ 80%

**Excellent**
- Test Accuracy ≥ 95%

With the threshold for each Performance defined, we will proceed to Deploy the Inception-like Model in the Code Cell below to determine the Test Accuracy.

In [ ]:
# ======================== Evaluation On Test Set ======================== #
test_loss_inc, test_acc_inc = model_inception_23_noaug_best.evaluate(test_generator_23, verbose=2)

# ======================== Creating DataFrame ======================== #
results_inc_df = pd.DataFrame({
    'Metric': ['Test Accuracy','Test Loss'],
    'Value':  [test_acc_inc, test_loss_inc]
})

# ======================== Displaying DataFrame ======================== #
results_inc_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to verify that our Inception-like Model has achieved a **Very Good** Test Accuracy of 87.6% and also a Relatively Low Test Loss of 0.59. This is a strong indicator that the Inception-like Model has very good Hyperparameters and will most likely perform well.

---
#### 4.4.9.4 Model's Accuracy Confusion Matrix

In this sub-section, we will be plotting the Confusion Matrix of the Inception-like Model so as to not only evaluate the Model's Accuracy, but also to identify potential Class Accuracy Weakness and Class Confusion. This will allow us to verify if any actions are required to address these issues and should there be a retuning that needs to be done.

In [ ]:
# ======================== Predicting On Test Data ======================== #
y_pred_inc = np.argmax(model_inception_23_noaug_best.predict(test_generator_23), axis=1)

# ======================== Plotting Confusion Matrix ======================== #
cm_inc = confusion_matrix(test_generator_23.classes, y_pred_inc)
disp_inc = ConfusionMatrixDisplay(cm_inc, display_labels=class_labels)

# ======================== Plot Configurations ======================== #
fig, ax = plt.subplots(figsize=(10,8))
disp_inc.plot(ax=ax, cmap='Blues', xticks_rotation=90)
plt.title("Confusion Matrix - Inception-like Model (23x23 No-Aug)")
plt.tight_layout()
plt.show()

With reference to the output above, we are able to see that the Inception-like Model performed relatively well as seen from the Dark Blue Coloured Squares running from top right to botton left of the Confusion Matrix. However, we are able to see that the Inception-like Model does occassionally mixes up some Classes with others such as 'Cabbage' with 'Cauliflower and Broccoli', 'Tomato' with 'Cauliflower and Broccoli' and 'Bitter_Gourd' and 'Brinjal' with 'Potato'. However, these Mix-ups are very minor and do not cause a major interference with the Test Accuracy.

---
#### 4.4.9.5 Model's Classification Report

In this sub-section, we will be running a Classification Report for the Inception-like Model performance. This Classification Report will give us insights on the various performance of the Inception-like Model for each Class instead of just an Overall Test Accuracy. Through this, we can evaluate the performance of the Inception-like Model on specific classes and also identify any Weak Classes. The formulas for the various metrics are indicated below:

---
**Precision**

Interpretation: Out of all Predicted Positives, how many are Actually Correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Where:**
- $TP$: True Positives  
- $FP$: False Positives

---
**Recall**

Interpretation: Out of all Actual Positives, how many did we Correctly Identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Where:*
- $TP$: True Positives
- $FN$: False Negatives

---
**F1-Score**

Interpretation: Balances Precision and Recall (useful when Classes are Imbalanced).
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---
**Support**

Interpretation: Total Number of Actual Samples in the Class.

$$
\text{Support} = TP + FN
$$

---
**Accuracy**

Interpretation: Overall Correct Predictions Divided by Total Samples.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Where:**
- $TP$: True Positives
- $TN$: True Negatives
- $FP$: False Positives
- $FN$: False Negatives

---
With the relevant formulas listed, we will proceed to produce the Classification Report in the Code Cell below.

In [ ]:
# ======================== Classification Report ======================== #
report_inc = classification_report(test_generator_23.classes, y_pred_inc,
                                   target_names=class_labels, output_dict=True)

# ======================== Rounding Off ======================== #
report_inc_df = pd.DataFrame(report_inc).transpose().round(4)

# ======================== Displaying of DataFrame ======================== #
report_inc_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame and the 'F1-Score' Data Column, we are able to see that the F1-Score is somewhat high (>75%) throughout the Classes with the exception of Tomato, Cabbage, and Cauliflower and Broccoli. This shows that the Inception-like Model generally performs quite well and consistent throughout all the Classes.

---
## 4.5 Depthwise Separable Model Training

In this section, we will be training Depthwise Separable which is a Convolutionary Neural Network (CNN) with the Data Input of **23px by 23px** and **No Data Augmentation**. We will be conducting various operations which includes but not limited to, Hyperparameter Tuning (KerasTuner) and Model Evaluation. We will also explore the various methods utilised to ensure the Model is tuned to output a high accuracy. The Formulas for the Depthwise Separable Model are indicated below:

---

**Depthwise Convolution:**
$$
z^{(l)}_{i,j,c} = \sum_{u=1}^{H_K} \sum_{v=1}^{W_K} W^{(l)}_{u,v,c} \cdot x^{(l-1)}_{i+u-1,\,j+v-1,\,c} + b^{(l)}_c
$$

Where:

- $z^{(l)}_{i,j,c}$: Output at Position $(i,j)$, Channel $c$, after Depthwise conv

- $x^{(l-1)}$: Input Feature Map from Previous Layer

- $W^{(l)}_{u,v,c}$: Depthwise Kernel for Channel $c$

- $b^{(l)}_c$: Bias for Channel $c$

- $H_K, W_K$: Height and Width of Kernel

- $c$: Operates per Channel (No Cross-channel Mixing)

---
**Pointwise Convolution (1×1 Convolution):**

$$
z^{(l)}_{i,j,k} = \sum_{c=1}^{C_{in}} W^{(l)}_{1\times1,c,k} \cdot a^{(l)}_{i,j,c} + b^{(l)}_k
$$

Where:

- $z^{(l)}_{i,j,k}$: Output Value at Position $(i,j)$, Output Channel $k$

- $a^{(l)}_{i,j,c}$: Activation from Depthwise Conv, Channel $c$

- $W^{(l)}_{1\times1,c,k}$: Pointwise Kernel from Input Channel $c$ to Output Channel $k$

- $b^{(l)}_k$: Bias for Output Channel $k$

- $C_{in}$: Number of Input Channels

---
**Combined Depthwise Separable Convolution**

$$
\text{SeparableConv} = \text{PointwiseConv}(\text{DepthwiseConv}(x))
$$

Where:

- $\text{SeparableConv}$: Final Output after Applying Both Depthwise and Pointwise Conv

- $\text{DepthwiseConv}(x)$: Applies a Unique Spatial Filter per Input Channel

- $\text{PointwiseConv}(\cdot)$: Applies $1 \times 1$ Conv to Mix Across Channels

---
**Final Softmax Output**

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}
$$
Where:

- $\hat{y}_i$: Probability of Class $i$

- $z_i$: Logit for Class $i$

- $K$: Total Number of Classes

---
**Categorical Cross-Entropy Loss**

$$
\mathcal{L} = -\sum_{i=1}^K y_i \cdot \log(\hat{y}_i)
$$
Where:

- $\mathcal{L}$: Cross-entropy Loss

- $y_i$: True Label (one-hot encoded)

- $\hat{y}_i$: Predicted Probability

- $K$: Number of Classes

---
With the Essential Formulas for CNN listed, we will proceed to start preparations for Training the Model in the Code Cells within this Section.

---
### 4.5.1 Defining Callbacks

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the model and also to increase the Model Accuracy and optimise the Computation Cost for training each model. These Callbacks will be main used during the Keras Tuner in the next sub-section. The formulas and logic for these Callbacks are indicated below:

---
**Learning Rate Scheduler:**

Purpose: Reduces Learning Rate During Training to Fine-tune Model Convergence.

$$\text{If } epoch \mod 10 = 0 \Rightarrow \eta_{new} = \frac{1}{2} \cdot \eta$$

Where:

- $\eta$ = current learning rate

- $\eta_{\text{new}}$ = updated learning rate

- $epoch \bmod 10$ checks if the epoch is a multiple of 10

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 3 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to Defind the Callbacks in the Code Cell below in preparation for Model Training.

In [ ]:
# ======================== Defining Learning Rate Scheduler ======================== #
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch:
        return lr * 0.5
    return lr

# ======================== Defining Callback ======================== #
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
lr_scheduler = LearningRateScheduler(scheduler)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure a high accuracy model training.

---
### 4.5.2 Defining Model Parameters

In this sub-section, we will be defining the Model's Parameters that will be tuned using Keras Tuner. There are various Parameters and Characteristics within the Code Cell below. All of these Parameters and their purpose have been indicated below:

---
**L2 Regularisation (Weight Decay)**

Purpose: Penalise Large Weights To Reduce Overfitting.

$$
\text{L2 Penalty} = \lambda \sum_{i=1}^{n} w_i^2
$$

**Where:**

- $λ$: Regularisation Coefficient (e.g., 0.001)

- $w_i$: Weight Parameter

---
**Convolutional Layer**

Purpose: Extract Spatial Features Using Learnable Filters.

$$
Y_{i,j}^{(k)} = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} W_{m,n}^{(k)} \cdot X_{i+m,j+n} + b^{(k)}
$$

**Where:**

- $Y_{i,j}^{(k)}$: Output feature map at position $(i,j)$

- $W^{(k)}$: Weights of the $k$-th Filter

- $X$: Input Patch

- $b^{(k)}$: Bias Term

---
**Batch Normalisation**

Purpose: Normalise Layer Activations to Speed Up & Stabilise Training.

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta
$$

**Where:**

- $x$: Activation Value
- $\mu$, $\sigma^2$: Batch Mean and Variance
- $\epsilon$: Small Constant for Numerical Stability
- $\gamma, \beta$: Trainable Parameters (scale and shift)

---
**MaxPooling2D**

Purpose: Reduces Spatial Dimensions while Retaining Important Features.

$$
Y_{i,j} = \max \{X_{2i,2j}, X_{2i,2j+1}, X_{2i+1,2j}, X_{2i+1,2j+1}\}
$$

**Where:**

- $X$: Input feature map
- $Y$: Pooled output

---
**Dropout**

Purpose: Randomly Disables Neurons to Reduce Overfitting.

$$
x_i' =
\begin{cases}
0 & \text{with probability } p \\
\frac{x_i}{1 - p} & \text{otherwise}
\end{cases}
$$

**Where:**

- $ x_i $: Input Neuron
- $ p $: Dropout Rate
- $ x_i' $: Output after Dropout

---
**Dense Layer (Fully Connected)**

Purpose: Learns Global Patterns after Spatial Feature Extraction.

$$
z = \sum_{i=1}^{n} w_i x_i + b, \quad a = \text{ReLU}(z)
$$

**Where:**

- $ w_i $: Weight
- $ x_i $: Input
- $ b $: Bias
- $ \text{ReLU}(z) = \max(0, z) $

---
**Softmax Output Layer**

Purpose: Converts Logits into Class Probabilities.

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**Where:**

- $ z_i $: Logit of Class $ i $
- $ K $: Number of Classes
- $ \sigma(z_i) $: Probability for Class $ i $

---
**Categorical Crossentropy Loss**

Purpose: Computes Distance Between Predicted and True Class Distributions.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

**Where:**

- $ y_i $: Actual Label (one-hot)
- $ \hat{y}_i $: Predicted Probability
- $ K $: Total Number of Classes

---

**Adam Optimiser**

Purpose: Combines Momentum and Adaptive Learning Rate, providing Fast Convergence with Stable Performance.

- First moment estimate:
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
$$

- Second moment estimate:
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left( \nabla L(\theta_t) \right)^2
$$

- Bias correction:
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

**Where:**

- $m_t$: First Moment Estimate (Mean of Gradients)  
- $v_t$: Second Moment Estimate (Uncentered Variance of Gradients)  
- $\hat{m}_t, \hat{v}_t$: Bias-corrected Estimates  
- $\beta_1, \beta_2$: Decay Rates (e.g., $0.9$, $0.999$)  
- $\nabla L(\theta_t)$: Gradient of the Loss at Step $t$  
- $\eta$: Learning Rate  
- $\epsilon$: Small Constant to Prevent Division by Zero  
- $\theta_t$: Model Weights at Step $t$  

---
**RMSprop Optimiser**

Purpose: Adapts Learning Rate Based on a Moving Average of Squared Gradients.

$$
\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)
$$

- Gradient moving average:
$$
v_t = \rho v_{t-1} + (1 - \rho) \left( \nabla L(\theta_t) \right)^2
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\nabla L(\theta_t)}{\sqrt{v_t} + \epsilon}
$$

**Where:**

- $v_t$: Running average of squared gradients  
- $\rho$: Decay factor (e.g., $0.9$)  
- $\nabla L(\theta_t)$: Gradient of the loss  
- $\eta$: Learning rate  
- $\epsilon$: Small constant  
- $\theta_t$: Model weights at step $t$  

---

**SGD with Momentum**

Purpose: Accelerates SGD by Accumulating a Velocity Vector in the Direction of Consistent Gradients.

- Velocity update:
$$
v_t = \mu v_{t-1} - \eta \nabla L(\theta_t)
$$

- Parameter update:
$$
\theta_{t+1} = \theta_t + v_t
$$

**Where:**

- $v_t$: Velocity (accumulated gradient)  
- $\mu$: Momentum coefficient (e.g., $0.9$)  
- $\eta$: Learning rate  
- $\nabla L(\theta_t)$: Gradient of loss  
- $\theta_t$: Model weights at step $t$  

---
With the formulas and purpose of each of the Parameter used for the Model listed, we will proceed to define the function in the cell below in preparation for the Model Hyperparameter Search using Keras Tuner.

In [ ]:
# ======================== Defining Model Parameters ======================== #
def build_ds_model(hp):
    # ----- Input ----- #
    inputs = Input(shape=(23, 23, 1))
    x = inputs

    # ----- Separable Conv Blocks ----- #
    for i in range(hp.Int("sep_blocks", 1, 3)):
        filters = hp.Choice(f"sep_filters_{i}", [16, 32, 64])
        x = SeparableConv2D(
            filters=filters,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            depthwise_regularizer=l2(0.001),
            pointwise_regularizer=l2(0.001)
        )(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D()(x)
        x = Dropout(hp.Float(f"sep_dropout_{i}", 0.1, 0.4, step=0.1))(x)

    x = Flatten()(x)

    # ----- Dense Layer ----- #
    dense_units = hp.Choice("dense_units", [64, 128, 256])
    x = Dense(dense_units,
              activation='relu',
              kernel_regularizer=l2(0.001))(x)
    x = Dropout(hp.Float("dense_dropout", 0.2, 0.5, step=0.1))(x)

    # ----- Output ----- #
    outputs = Dense(11, activation='softmax')(x)
    model = Model(inputs, outputs)

    # ----- Optimiser ----- #
    opt_choice = hp.Choice("optimizer", ['adam', 'rmsprop', 'sgd'])
    lr_choice  = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    if opt_choice == 'adam':
        optimizer = Adam(lr_choice)
    elif opt_choice == 'rmsprop':
        optimizer = RMSprop(lr_choice)
    else:
        optimizer = SGD(lr_choice, momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

With reference to the Code Cell above, we are able to confirm that the Function for Creating a New Model for the input of 23px by 23px have been successfully defined and it can be utilised for the Keras Tuner.

---
### 4.5.3 Setting Up Keras Tuner

In this sub-section, we will be setting up the Keras Tuner which will be later utilised to find the best hyperparameter for the CNN Model. Setting up the Keras Tuner appropriately will also ensure that the Random Search do not need to be reruned again as a directory can be set up to link the saved Searches from the Google Drive. This will save up Computational Cost and ensure a more efficient and consistent rerun of Codes should the need arise. To allow for replication, the Seed will be set to 42. However, due to varying Computational Capabilities of different GPUs and/or TPUs, a slight deviation will be normal in the various Evaluation Metrics despite having the same Seed.

In [ ]:
# ======================== Setting Up Keras Tuner ======================== #
tuner_ds = kt.RandomSearch(
    build_ds_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='Tunings',
    project_name='kerastuner_23_noaug_ds',
    overwrite=False,
    seed=42
)

With reference to the Code Cell above, we are able to verify that the Keras Tuner have been established correctly with the directory defined successfully too. We shall move on to conduct the Random Search using the Keras Tuner.

---
### 4.5.4 Conducting Keras Tuner Random Search

In this sub-section, we will be conducting the Random Search to find the best Hyperparameter for the CNN Model using the Keras Tuner. Unlike traditional AIML, there will only be a Random Search and no Grid Search due to the extremely high Computational Resources Require to run a Grid Search for Deep Learning. It is also important to note that the Hyperparameter Random Search will take a long duration due to the nature of the Deep Learning.

In [ ]:
# ======================== Random Searching For Best Hyperparameters ======================== #
tuner_ds.search(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output above, we are able to see that the best Validation Accuracy obtained in 0.908 and the total time taken to conduct the Keras Tuner to find the Best Hyperparameter is around 5 Hours. As the data from the search have been saved, we will not have to rerun the Keras Tuner for this Model again.

---
### 4.5.5 Obtaining Best Hyperparameter

In this sub-section, we will be obtaining the Best Hyperparameter from the Keras Tuner Random Search so as to prepare for the training of the Depthwise Separable Model in the next sub-section.

In [ ]:
# ======================== Retrieving Best Model ======================== #
best_hp_ds = tuner_ds.get_best_hyperparameters(1)[0]
model_ds_23_noaug_best = tuner_ds.hypermodel.build(best_hp_ds)

With reference to the Code Cell above, we are able to confirm that the Best Hyperparameter have been obtained and the Model Architecture have been successfully built.

---
### 4.5.6 Depthwise Separable Model Checkpoint

In this sub-section, we will be establishing the Model Checkpoint and saving the Model to our Google Drive. It is extremely important to take note that we will only be saving the Best Model so as to prevent excessive storage usage. The key monitor that will be used is the Validation Accuracy so as to achieve the Ideal Fit when placed together with the Validation Data. We will also be only saving the Best Model so as to prevent excessive space usage.

In [ ]:
# ======================== Checkpoint & Training ======================== #
checkpoint_ds = ModelCheckpoint(
    filepath=os.path.join(model_dir, "best_model_23_noaug_ds_tuned.keras"),
    monitor='val_accuracy', save_best_only=True
)

With reference to the Code Cell above, we are able to confirm that the Model have been saved successfully to the Google Drive and the Model is ready for Deployment should the need arise.

---
### 4.5.7 Training Depthwise Separable Model

In this sub-section, we will be training the Best Depthwise Separable Model using the extracted Hyperparameter. This will subsequently allow us to obtain the Model's History which can allow us to conduct various visualisations such as plotting out the Learning Curves to identify the fitting of the Model.

In [ ]:
# ======================== Training Best Model ======================== #
history_ds_23_noaug_best = model_ds_23_noaug_best.fit(
    train_generator_23,
    validation_data=val_generator_23,
    epochs=100,
    callbacks=[early_stop, reduce_lr, lr_scheduler, checkpoint_ds],
    class_weight=class_weight_dict,
    verbose=2
)

With reference to the output of the Code Cell above, we are able to infer that the Depthwise Separable Model have been successfully trained and will now be ready for Model Evaluation and Model Deployment on the Test Dataset.

---
### 4.5.8 Saving of Model Weights, Model Architecture and Model Tunings

In this sub-section, we will be Saving the Model's Weights into our Google Drive as per the expectaions of the Assessment. The weight will be saved in 'h5' format and will be submitted. The Tunings Record and the Model Architecture will also be saved so as to prevent the need for re-running all the previous codes which will be extremely Computationally Expensive.

In [ ]:
# ==================== Copy Results Back to Drive ==================== #
model_ds_23_noaug_best.save_weights("Weights/weights_23_noaug_ds_tuned.weights.h5")

shutil.copytree('Tunings',
                os.path.join(DRIVE_ROOT, 'Tunings'),
                dirs_exist_ok=True)

shutil.copytree('Models',
                os.path.join(DRIVE_ROOT, 'Models'),
                dirs_exist_ok=True)

shutil.copytree('Weights',
                os.path.join(DRIVE_ROOT, 'Weights'),
                dirs_exist_ok=True)

With reference to the Code Cell above, we are able to verify that the Depthwise Separable Model's Weight have been saved successfully to the Google Drive.

---
### 4.5.9 Model Evaluations

In this sub-section, we will be evaluating the Depthwise Separable Model's Performance on the 23px by 23px Data (Not Augmented). We will not be evaluating solely based of the Model's Accuracy, but also based of it's Learning Curve and Specific Images Accuracy. This is to ensure that the Model is able to perform well at a hollistic level and not just solely based of Accuracy or Class Imbalanced Accuracy. More details will be established for the various Evaluations.

---
#### 4.5.9.1 Plotting of Learning Curve

In this sub-section, we will be plotting the Learning Curve to identify the Learning Capabilities of the Depthwise Separable Model. This will allow us to identify if the Model is Underfitting, Overfitting or have attained the Ideal Fit. Should the Depthwise Separable Model be Underfitting, it means that the Depthwise Separable Model is failing to identify and learn the various patterns of the images. Should the Depthwise Separable Model be Overfitting, it means that the Model has attained a High Accuracy solely based of Memorising the Data instead of Learning the Data's Attributes. However, if the Depthwise Separable Model has a Ideal Fit it is a very good indicator that the Depthwise Separable Model has Learnt from the Training Data. The Potential Observations and subsequent Action Plans are indicated below:

**Under Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Over Fitting Learning Curve**
- Re-run Keras Tuner to get Better Hyperparameters
- Adjust the Patience of Early Stopping
- Adjust the Number of Epochs to Obtain Better Learning Rate

**Ideal Fitting Learning Curve**
- Accept the Hyperparameter
- Deploy the Model on Testing Data

With the Potential Observations and Action Plan listed, we will proceed to plot out the Learning Curve in the Code Cell below.

In [ ]:
def plot_history(history, title="Depthwise Separable CNN: 23x23 No-Aug"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # ----- Accuracy Plot ----- #
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # ----- Loss Plot ------ #
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# ======================== Plotting ======================== #
plot_history(history_ds_23_noaug_best, title="Depthwise Separable CNN: 23x23 No-Aug")

With reference to the output of the Learning Curve above, we are able to see that the Depthwise Separable Model has achieved a **Ideal Learning Curve**. This means that the Depthwise Separable Model has indeed learnt from the Training Dataset instead of Memorising or Not Learning at all. With this observation stated, we will now proceed to View the Hyperparameter of the Depthwise Separable Model and subsequently, Deploy the Depthwise Separable Model on the Testing Data to get the final Evaluation Metrics.

---
#### 4.5.9.2 Model Hyperparameter

In this sub-section, we will be extracting out the Depthwise Separable Model's Hyperparameter so as to display the current and Best Hyperparameter for future references within the Project. It is important to take note that the Hyperparameter may be different for other users of the code due to the varying Computational Environment.

In [ ]:
# ======================== Hyperparameter DataFrame ======================== #
best_hps_ds_dict = best_hp_ds.values

# ======================== Creating of DataFrame ======================== #
hp_ds_df = pd.DataFrame(list(best_hps_ds_dict.items()), columns=['Hyperparameter','Value'])

# ======================== Displaying of DataFrame ======================== #
hp_ds_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to view the various parameters of the Depthwise Separable Model and we are able to identify the various layers and neurons that makes up the Hidden Layer of the Depthwise Separable Model.

---
#### 4.5.9.3 Model Deployment on Testing Data

In this sub-section, we will be Deploying our Depthwise Separable Model on the Testing Data provided. Through the Deployment, we are able to obtain a Test Accuracy Metric and also the Test Loss Metric which will allow us to verify the Depthwise Separable Model's performance on a Protected Dataset to simulate real-world deployments. The pre-requisite of the Model Performance is evaluated on the following Test Accuracy Threshold:

**Poor**
- Test Accuracy ≤ 9.10...%

**Acceptable**
- Test Accuracy ≥ 20%

**Good**
- Test Accuracy ≥ 50%

**Very Good**
- Test Accuracy ≥ 80%

**Excellent**
- Test Accuracy ≥ 95%

With the threshold for each Performance defined, we will proceed to Deploy the Depthwise Separable Model in the Code Cell below to determine the Test Accuracy.

In [ ]:
# ======================== Evaluating On Test Set ======================== #
test_loss_ds, test_acc_ds = model_ds_23_noaug_best.evaluate(test_generator_23, verbose=2)

# ======================== Creating DataFrame ======================== #
results_ds_df = pd.DataFrame({
    'Metric': ['Test Accuracy','Test Loss'],
    'Value':  [test_acc_ds, test_loss_ds]
})

# ======================== Displaying of DataFrame ======================== #
results_ds_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame above, we are able to verify that our Depthwise Separable Model has achieved a **Very Good** Test Accuracy of 87.6% and also a Relatively Low Test Loss of 0.59. This is a strong indicator that the Depthwise Separable Model has very good Hyperparameters and will most likely perform well.

---
#### 4.5.9.4 Model's Accuracy Confusion Matrix

In this sub-section, we will be plotting the Confusion Matrix of the Depthwise Separable Model so as to not only evaluate the Model's Accuracy, but also to identify potential Class Accuracy Weakness and Class Confusion. This will allow us to verify if any actions are required to address these issues and should there be a retuning that needs to be done.

In [ ]:
# ======================== Predicting On Test Data ======================== #
y_pred_ds = np.argmax(model_ds_23_noaug_best.predict(test_generator_23), axis=1)

# ======================== Plotting Confusion Matrix ======================== #
cm_ds = confusion_matrix(test_generator_23.classes, y_pred_ds)
disp_ds = ConfusionMatrixDisplay(cm_ds, display_labels=class_labels)

# ======================== Plot Configurations ======================== #
fig, ax = plt.subplots(figsize=(10,8))
disp_ds.plot(ax=ax, cmap='Blues', xticks_rotation=90)
plt.title("Confusion Matrix - Depthwise Separable CNN (23x23 No-Aug)")
plt.tight_layout()
plt.show()

With reference to the output above, we are able to see that the Depthwise Separable Model performed relatively well as seen from the Dark Blue Coloured Squares running from top right to botton left of the Confusion Matrix. However, we are able to see that the Depthwise Separable Model does occassionally mixes up some Classes with others such as 'Cabbage' with 'Cauliflower and Broccoli', 'Tomato' with 'Cauliflower and Broccoli' and 'Bitter_Gourd'. However, these Mix-ups are very minor and do not cause a major interference with the Test Accuracy.

---
#### 4.5.9.5 Model's Classification Report

In this sub-section, we will be running a Classification Report for the Depthwise Depthwise Separable Model performance. This Classification Report will give us insights on the various performance of the Depthwise Depthwise Separable Model for each Class instead of just an Overall Test Accuracy. Through this, we can evaluate the performance of the Sequential Model on specific classes and also identify any Weak Classes. The formulas for the various metrics are indicated below:

---
**Precision**

Interpretation: Out of all Predicted Positives, how many are Actually Correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Where:**
- $TP$: True Positives  
- $FP$: False Positives

---
**Recall**

Interpretation: Out of all Actual Positives, how many did we Correctly Identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**Where:*
- $TP$: True Positives
- $FN$: False Negatives

---
**F1-Score**

Interpretation: Balances Precision and Recall (useful when Classes are Imbalanced).
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---
**Support**

Interpretation: Total Number of Actual Samples in the Class.

$$
\text{Support} = TP + FN
$$

---
**Accuracy**

Interpretation: Overall Correct Predictions Divided by Total Samples.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Where:**
- $TP$: True Positives
- $TN$: True Negatives
- $FP$: False Positives
- $FN$: False Negatives

---
With the relevant formulas listed, we will proceed to produce the Classification Report in the Code Cell below.

In [ ]:
# ======================== Generating Classification Report ======================== #
report_ds = classification_report(test_generator_23.classes, y_pred_ds,
                                  target_names=class_labels, output_dict=True)

# ======================== Rounding Off ======================== #
report_ds_df = pd.DataFrame(report_ds).transpose().round(4)

# ======================== Displaying of DataFrame ======================== #
report_ds_df.style.background_gradient(cmap="Blues")

With reference to the output of the DataFrame and the 'F1-Score' Data Column, we are able to see that the F1-Score is somewhat high (>75%) throughout the Classes with the exception of Cauliflower and Broccoli, Cabbage, and Tomato. This shows that the Depthwise Separable Model generally performs quite well and consistent throughout all the Classes.